# Full training: original random eight-layer RoBERTa

**Author: Mohammad Albatayneh.** The original individual project code was written
by Mohammad, with AI assistance correcting errors. This notebook packages the
repaired experiment for Google Colab; it does not claim a completed full GPU run.

Open this `.ipynb` in [Google Colab](https://colab.research.google.com/), select
**Runtime → Change runtime type → GPU**, then run the cells in order. You need the
original **`train_preprocessed.csv`**, Python **3.12+**, CUDA-enabled torch **2.8+**,
and enough Google Drive space for private checkpoints and model exports.

All executable project source is visibly embedded below. No published repository,
Git clone, API token, pretrained encoder weights, or paid service is required.
Only the public `roberta-base` **tokenizer** is downloaded. A free Colab GPU's
availability, memory, and session duration are not guaranteed.

The normal workflow trains the full eligible cohort and returns
**`results_for_review.zip`**. Keep the optional internal-test switch **False** for
the first results review. Do not upload the unlabeled competition test CSV.

## Fixed experiment and data boundary

| Setting | Value |
|---|---|
| Source | Original supplied `train_preprocessed.csv`, SHA-256 checked |
| Input / eligible rows | 159,571 / 159,564; explicitly drop seven empty texts |
| Conflicting annotations | Retain all 93 conflicting duplicate groups |
| Split | Seed 42; normalized text groups kept intact; nominal 70/15/15 |
| Actual train / validation / internal test | 111,731 / 23,878 / 23,955 |
| Targets, in order | `identity_hate`, `insult`, `obscene`, `severe_toxic`, `threat`, `toxic` |
| Excluded derived column | `toxicity` is a target sum; never an input feature |
| Random encoder | 8 layers, hidden 512, 8 heads, intermediate 2,048 |
| Tokenizer / sequence length | Unmodified `roberta-base` / 256 tokens |
| Parameters | 51,484,678; model weights initialized randomly |
| Optimizer / schedule | AdamW, learning rate 4e-5, weight decay 0.01; cosine, 10% warmup |
| Training | At most five epochs; patience three; weighted BCE, train-only weights |
| Batches / precision | Microbatch 8 × accumulation 32; CUDA float16 AMP |
| Stability | Zero workers; deterministic algorithms; gradient norm clip 1.0 |
| Selection | Minimum validation loss; reload/export best checkpoint; fresh validation |
| Thresholds | Six validation-selected F1 thresholds, then frozen |

The CSV column order need not match this runtime order: labels are selected by
column name and every artifact records the explicit order above.

Validation scores are **selection diagnostics**. The internal exploratory test
stays unscored by default and is not a pristine external test: this cohort was
examined historically. The competition test CSV has no labels and cannot be scored.

## 1. Check the GPU runtime and create the temporary source folder

This cell refuses a CPU-only runtime. It preserves the torch CUDA build already
installed by Colab. Environment variables are set before CUDA is initialized.

In [ ]:
import importlib.metadata
import os
import sys
from pathlib import Path
from packaging.version import Version

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if sys.version_info < (3, 12):
    raise RuntimeError("Select a Python 3.12+ Colab runtime before installing these pins.")
import torch
if Version(torch.__version__.split("+")[0]) < Version("2.8.0"):
    raise RuntimeError("Use a Colab runtime with CUDA-enabled torch >=2.8; do not install a CPU wheel.")
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, reconnect, and rerun this cell.")
COLAB_TORCH_VERSION = torch.__version__
COLAB_TORCH_DISTRIBUTION_VERSION = importlib.metadata.version("torch")
PROJECT_DIR = Path("/content/toxic-comment-classification")
for relative in ("src/toxic_comments", "scripts"):
    (PROJECT_DIR / relative).mkdir(parents=True, exist_ok=True)
print({"python": sys.version.split()[0], "torch": COLAB_TORCH_VERSION,
       "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)})


## 2. Materialize the reviewed source

Run all eight visible `%%writefile` cells. They reproduce the exact repository
source bytes; the following SHA-256 manifest checks every embedded file.

In [ ]:
%%writefile /content/toxic-comment-classification/src/toxic_comments/__init__.py
"""Reproducible six-label toxic comment experiments."""

from .data import LABELS

__all__ = ["LABELS"]
__version__ = "0.1.0"


In [ ]:
%%writefile /content/toxic-comment-classification/src/toxic_comments/artifacts.py
"""Private run artifacts, integrity checks, and a single-use test boundary.

Checksums detect accidental changes; this is not a tamper-proof security system.
Model deserialization is permitted only for an intact run created locally.
"""

from __future__ import annotations

import importlib.metadata
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np

from .data import LABELS, SplitIndices, file_sha256
from .metrics import _validate_arrays, _validate_thresholds

MANIFEST_SCHEMA = 1
TEST_MARKER = ".test_evaluation_started"


def _json_default(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")


def write_json(path: str | Path, value: Any) -> Path:
    """Create a JSON artifact once; NaN/Infinity are always rejected."""
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    serialized = json.dumps(value, indent=2, sort_keys=True, allow_nan=False, default=_json_default) + "\n"
    with destination.open("x", encoding="utf-8") as stream:
        stream.write(serialized)
    return destination


def read_json(path: str | Path) -> Any:
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)


def create_run_directory(path: str | Path) -> Path:
    """Refuse overwrite, and require ignored storage for in-repository runs.

    Temporary directories outside the repository support isolated tests. Normal
    project runs belong under ``runs/`` or ``private/``, both ignored by git.
    """
    destination = Path(path).resolve()
    repository = Path(__file__).resolve().parents[2]
    if destination.is_relative_to(repository):
        relative = destination.relative_to(repository)
        if not relative.parts or relative.parts[0] not in {"runs", "private"} or len(relative.parts) < 2:
            raise ValueError("Run directories inside the repository must be under ignored runs/ or private/")
    destination.mkdir(parents=True, exist_ok=False)
    (destination / "private").mkdir()
    return destination


def resolve_run_path(run_dir: str | Path, stored_path: str) -> Path:
    candidate = Path(stored_path)
    return candidate.resolve() if candidate.is_absolute() else (Path(run_dir) / candidate).resolve()


def _relative_private_path(run_dir: Path, path: Path) -> str:
    resolved = path.resolve()
    if not resolved.is_relative_to(run_dir) or resolved == run_dir:
        raise ValueError("Frozen model and artifact files must be inside the run directory")
    return resolved.relative_to(run_dir).as_posix()


def _code_hashes() -> dict[str, str]:
    repository = Path(__file__).resolve().parents[2]
    return {
        path.relative_to(repository).as_posix(): file_sha256(path)
        for path in sorted(Path(__file__).parent.glob("*.py"))
    }


def runtime_metadata(seed: int) -> dict[str, Any]:
    """Capture the executable environment and source state as private evidence."""
    repository = Path(__file__).resolve().parents[2]

    def git(*arguments: str) -> str | None:
        try:
            result = subprocess.run(
                ["git", "-C", str(repository), *arguments], capture_output=True, text=True, check=False
            )
        except FileNotFoundError:
            return None
        return result.stdout.strip() if result.returncode == 0 else None

    versions = {}
    for name in ("numpy", "pandas", "scipy", "scikit-learn", "joblib", "torch", "transformers"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    git_status = git("status", "--porcelain")
    return {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "dependencies": versions,
        "seed": int(seed),
        "python_hash_seed": os.environ.get("PYTHONHASHSEED"),
        "thread_environment": {
            name: os.environ.get(name)
            for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
        },
        "git_revision": git("rev-parse", "HEAD"),
        "working_tree_dirty": bool(git_status) if git_status is not None else None,
        "code_sha256": _code_hashes(),
    }


def _validated_splits(splits: SplitIndices | Mapping[str, np.ndarray], rows: int) -> dict[str, np.ndarray]:
    mapping = splits.as_dict() if isinstance(splits, SplitIndices) else dict(splits)
    if set(mapping) != {"train", "validation", "test"}:
        raise ValueError("Splits must contain exactly train, validation, and test")
    clean = {}
    for name, indices in mapping.items():
        values = np.asarray(indices)
        if values.ndim != 1 or not len(values) or not np.issubdtype(values.dtype, np.integer):
            raise ValueError(f"{name} split must contain a nonempty integer index array")
        clean[name] = values.astype(np.int64)
    combined = np.concatenate(list(clean.values()))
    if len(combined) != rows or not np.array_equal(np.sort(combined), np.arange(rows)):
        raise ValueError("Splits must partition every retained row exactly once")
    return clean


def freeze_run(
    run_dir: str | Path,
    *,
    config: dict[str, Any],
    data_summary: dict[str, Any],
    source_path: str | Path,
    model_path: str | Path,
    thresholds: Sequence[float],
    splits: SplitIndices | Mapping[str, np.ndarray],
    metadata: dict[str, Any],
    extra_artifacts: Sequence[str | Path] = (),
) -> dict[str, Any]:
    """Freeze model, input bytes, split, thresholds, configuration and metadata.

    ``model_path`` and extra artifacts can be files or directories. Paths passed
    by callers should be absolute or relative to their current working directory.
    The manifest stores paths inside the run relative to it.
    """
    root = Path(run_dir).resolve()
    if not root.is_dir():
        raise ValueError("Create a new run directory before freezing it")
    if (root / "manifest.json").exists() or (root / TEST_MARKER).exists():
        raise FileExistsError("Run is already frozen or test evaluation has started")
    threshold_values = _validate_thresholds(thresholds)
    indices = _validated_splits(splits, int(data_summary["retained_rows"]))
    model = Path(model_path).resolve()
    model_relative = _relative_private_path(root, model)
    if not model.exists():
        raise FileNotFoundError(model)
    source = Path(source_path).resolve()
    source_hash = file_sha256(source)
    source_stored = source.relative_to(root).as_posix() if source.is_relative_to(root) else str(source)
    code_hashes = _code_hashes()
    if "code_sha256" in metadata and metadata["code_sha256"] != code_hashes:
        raise ValueError("Source code changed during development; start a new run after completing code changes")
    metadata = {
        **metadata,
        "code_sha256": code_hashes,
        "code_integrity_policy": "Changes to the recorded source files or source-file set invalidate this run",
    }
    initial = [
        write_json(root / "config.json", config),
        write_json(root / "data_summary.json", data_summary),
        write_json(root / "metadata.json", metadata),
        write_json(root / "thresholds.json", {"label_order": list(LABELS), "thresholds": threshold_values}),
    ]
    split_path = root / "private" / "splits.npz"
    split_path.parent.mkdir(parents=True, exist_ok=True)
    with split_path.open("xb") as stream:
        np.savez_compressed(stream, **indices)
    initial.append(split_path)
    files: dict[str, str] = {}
    for artifact in [*initial, model, *map(Path, extra_artifacts)]:
        if not artifact.exists():
            raise FileNotFoundError(artifact)
        candidates = sorted(artifact.rglob("*")) if artifact.is_dir() else [artifact]
        for candidate in candidates:
            if candidate.is_file():
                relative = _relative_private_path(root, candidate)
                files[relative] = file_sha256(candidate)
    if model.is_dir() and not any(name.startswith(model_relative + "/") for name in files):
        raise ValueError("Model directory must contain at least one file")
    manifest = {
        "schema_version": MANIFEST_SCHEMA,
        "label_order": list(LABELS),
        "source_path": source_stored,
        "source_sha256": source_hash,
        "model_path": model_relative,
        "artifacts_sha256": files,
        "test_policy": "single evaluation attempt; thresholds and model frozen before test",
    }
    manifest_path = write_json(root / "manifest.json", manifest)
    with (root / "manifest.sha256").open("x", encoding="ascii") as stream:
        stream.write(file_sha256(manifest_path) + "\n")
    return manifest


def verify_frozen_run(run_dir: str | Path, *, source_path: str | Path | None = None) -> dict[str, Any]:
    """Verify the frozen manifest, every artifact and the selected input bytes."""
    root = Path(run_dir).resolve()
    manifest_path = root / "manifest.json"
    expected_manifest_hash = (root / "manifest.sha256").read_text(encoding="ascii").strip()
    if file_sha256(manifest_path) != expected_manifest_hash:
        raise ValueError("Frozen manifest integrity check failed")
    manifest = read_json(manifest_path)
    if manifest.get("schema_version") != MANIFEST_SCHEMA or manifest.get("label_order") != list(LABELS):
        raise ValueError("Frozen run schema or label order is incompatible")
    for name, expected in manifest["artifacts_sha256"].items():
        candidate = resolve_run_path(root, name)
        if not candidate.is_relative_to(root) or not candidate.is_file() or file_sha256(candidate) != expected:
            raise ValueError(f"Frozen artifact integrity check failed: {name}")
    model = resolve_run_path(root, manifest["model_path"])
    if not model.is_relative_to(root) or not model.exists():
        raise ValueError("Frozen model must remain inside the verified run directory")
    model_relative = model.relative_to(root).as_posix()
    if model.is_file():
        recorded_model = model_relative in manifest["artifacts_sha256"]
    else:
        recorded_files = {
            name for name in manifest["artifacts_sha256"] if name.startswith(model_relative + "/")
        }
        actual_files = {
            _relative_private_path(root, candidate)
            for candidate in model.rglob("*") if candidate.is_file()
        }
        recorded_model = bool(recorded_files) and recorded_files == actual_files
    if not recorded_model:
        raise ValueError("Frozen model file set is not covered by the artifact checksums")
    recorded_code = read_json(root / "metadata.json")["code_sha256"]
    actual_code = _code_hashes()
    if recorded_code != actual_code:
        changed = sorted(
            name for name in set(recorded_code) | set(actual_code)
            if recorded_code.get(name) != actual_code.get(name)
        )
        raise ValueError(f"Source code changed after development; frozen run is invalid: {', '.join(changed)}")
    source = Path(source_path).resolve() if source_path is not None else resolve_run_path(root, manifest["source_path"])
    if not source.is_file() or file_sha256(source) != manifest["source_sha256"]:
        raise ValueError("Frozen source dataset integrity check failed")
    return manifest


def load_splits(run_dir: str | Path) -> SplitIndices:
    root = Path(run_dir)
    rows = int(read_json(root / "data_summary.json")["retained_rows"])
    with np.load(root / "private" / "splits.npz", allow_pickle=False) as archive:
        arrays = _validated_splits({name: archive[name] for name in archive.files}, rows)
    return SplitIndices(arrays["train"], arrays["validation"], arrays["test"])


def load_thresholds(run_dir: str | Path) -> np.ndarray:
    value = read_json(Path(run_dir) / "thresholds.json")
    if value["label_order"] != list(LABELS):
        raise ValueError("Threshold label order is incompatible")
    return _validate_thresholds(value["thresholds"])


def begin_test_evaluation(run_dir: str | Path, *, source_path: str | Path | None = None) -> Path:
    """Claim the single test attempt before any test inference or metric access.

    A failed attempt remains claimed, so rerunning cannot become silent test
    tuning. No reset or override is provided by the CLI.
    """
    root = Path(run_dir)
    if not (root / "manifest.sha256").is_file():
        raise ValueError("Test evaluation requires a frozen run")
    verify_frozen_run(root, source_path=source_path)
    if (root / "public_test_results.json").exists():
        raise FileExistsError("Test results already exist; evaluation cannot be repeated")
    return write_json(root / TEST_MARKER, {
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "manifest_sha256": file_sha256(root / "manifest.json"),
    })


def _check_public_aggregates(value: Any) -> None:
    forbidden = {
        "id", "ids", "row_id", "row_ids", "row_indices", "comment_text", "text",
        "texts", "comments", "raw_comments", "rawcomments", "raw_text", "raw_texts",
        "records", "targets", "y_true", "predictions", "y_pred", "probabilities",
        "scores", "y_score", "logits", "features", "embeddings", "source_path", "model_path",
    }
    if isinstance(value, dict):
        for key, child in value.items():
            lowered = str(key).lower()
            if lowered in forbidden or "sha256" in lowered or "hash" in lowered:
                raise ValueError(f"Private field cannot be included in public results: {key}")
            _check_public_aggregates(child)
    elif isinstance(value, (list, tuple)):
        for child in value:
            _check_public_aggregates(child)


def complete_test_evaluation(
    run_dir: str | Path,
    *,
    public_results: dict[str, Any],
    targets: np.ndarray,
    probabilities: np.ndarray,
    row_indices: np.ndarray,
    source_path: str | Path | None = None,
) -> Path:
    root = Path(run_dir)
    if not (root / TEST_MARKER).exists():
        raise ValueError("Begin the single test evaluation before writing its results")
    if (root / "public_test_results.json").exists():
        raise FileExistsError("Test results already exist; evaluation cannot be repeated")
    # Recheck after inference as well as before it. A code/model/input change
    # during an attempt must not produce a result attributed to frozen bytes.
    verify_frozen_run(root, source_path=source_path)
    claim = read_json(root / TEST_MARKER)
    if claim.get("manifest_sha256") != file_sha256(root / "manifest.json"):
        raise ValueError("Test attempt does not match the frozen manifest")
    clean_targets, clean_scores = _validate_arrays(targets, probabilities)
    indices = np.asarray(row_indices)
    if not np.array_equal(indices, load_splits(root).test) or len(indices) != len(clean_targets):
        raise ValueError("Test predictions must match the frozen test indices in order")
    _check_public_aggregates(public_results)
    with (root / "private" / "test_predictions.npz").open("xb") as stream:
        np.savez_compressed(stream, row_indices=indices, targets=clean_targets, probabilities=clean_scores)
    return write_json(root / "public_test_results.json", public_results)


In [ ]:
%%writefile /content/toxic-comment-classification/src/toxic_comments/baseline.py
"""CPU reference: train-only word TF-IDF and six balanced logistic regressions.

Development uses training and validation data, then freezes the model and
validation-selected thresholds. Test evaluation is a separate, single-use
operation. The offline smoke command uses an explicitly synthetic fixture.
"""

from __future__ import annotations

import argparse
import json
import shutil
import tempfile
import warnings
from pathlib import Path
from typing import Any

import joblib
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

from .artifacts import (
    begin_test_evaluation,
    complete_test_evaluation,
    create_run_directory,
    freeze_run,
    load_splits,
    load_thresholds,
    read_json,
    resolve_run_path,
    runtime_metadata,
    verify_frozen_run,
    write_json,
)
from .data import (
    LABELS,
    create_synthetic_fixture,
    file_sha256,
    load_dataset,
    split_dataset,
    split_summary,
)
from .metrics import DEFAULT_THRESHOLD_GRID, _validate_arrays, evaluate_predictions, select_thresholds

METHOD = "tfidf_logistic_regression"
PROTOCOL_NOTE = (
    "New internal exploratory holdout; the historical cohort was previously examined, "
    "so this is not a pristine external test."
)


def _validate_seed(seed: int) -> int:
    if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    return int(seed)


def build_pipeline(*, seed: int = 42) -> Pipeline:
    """Construct the fixed reference model; fitting never happens here."""
    seed = _validate_seed(seed)
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50_000,
            sublinear_tf=True, lowercase=True, norm="l2", dtype=np.float64,
        )),
        ("classifier", OneVsRestClassifier(
            LogisticRegression(
                solver="liblinear", C=4.0, class_weight="balanced", max_iter=1000,
                tol=1e-4, random_state=seed,
            ),
            n_jobs=1,
        )),
    ])


def validate_training_targets(targets: np.ndarray) -> None:
    """Reject unsupported train labels instead of fitting constant predictors."""
    values = np.asarray(targets)
    if values.ndim != 2 or values.shape[1] != len(LABELS) or not len(values):
        raise ValueError(f"Training targets must have a nonempty (n, {len(LABELS)}) shape")
    if not np.isin(values, [0, 1]).all():
        raise ValueError("Training targets must be binary 0/1 values")
    missing = [label for column, label in enumerate(LABELS) if len(np.unique(values[:, column])) != 2]
    if missing:
        raise ValueError(f"Training requires both positive and negative examples for every label: {', '.join(missing)}")


def _validate_model(model: Any) -> Pipeline:
    if not isinstance(model, Pipeline) or set(model.named_steps) != {"tfidf", "classifier"}:
        raise ValueError("Frozen model is not the expected TF-IDF/logistic-regression pipeline")
    vectorizer = model.named_steps["tfidf"]
    classifier = model.named_steps["classifier"]
    if not isinstance(vectorizer, TfidfVectorizer) or not isinstance(classifier, OneVsRestClassifier):
        raise ValueError("Frozen model contains incompatible estimator types")
    if tuple(getattr(model, "toxic_label_order_", ())) != LABELS:
        raise ValueError("Frozen model label order is incompatible")
    if not getattr(classifier, "multilabel_", False) or len(classifier.estimators_) != len(LABELS):
        raise ValueError("Frozen classifier must contain six fitted multilabel estimators")
    if not np.array_equal(classifier.classes_, np.arange(len(LABELS))):
        raise ValueError("Frozen classifier column order is incompatible")
    if any(not np.array_equal(estimator.classes_, [0, 1]) for estimator in classifier.estimators_):
        raise ValueError("Each frozen binary estimator must expose class-1 probabilities")
    return model


def run_development(
    data_path: str | Path,
    run_dir: str | Path,
    *,
    seed: int = 42,
    conflict_policy: str = "error",
    empty_text_policy: str = "error",
    quiet: bool = False,
) -> dict[str, Any]:
    """Fit on train, select thresholds on validation, export and freeze once.

    A private, byte-identical source snapshot makes subsequent evaluation
    independent of the caller's original file location. A failed development
    attempt leaves its directory in place and requires a new directory to retry.
    """
    del quiet  # The public API matches neural development; this fit has no epoch log.
    seed = _validate_seed(seed)
    if conflict_policy not in {"error", "keep"}:
        raise ValueError("conflict_policy must be 'error' or 'keep'")
    if empty_text_policy not in {"error", "drop"}:
        raise ValueError("empty_text_policy must be 'error' or 'drop'")
    source = Path(data_path).resolve()
    source_hash = file_sha256(source)
    destination = create_run_directory(run_dir)
    metadata = runtime_metadata(seed)
    source_snapshot = destination / "private" / "source.csv"
    shutil.copyfile(source, source_snapshot)
    dataset = load_dataset(
        source_snapshot, conflict_policy=conflict_policy, empty_text_policy=empty_text_policy,
    )
    if dataset.source_sha256 != source_hash or file_sha256(source) != source_hash:
        raise ValueError("Source dataset changed while its private snapshot was being created")
    splits = split_dataset(dataset, seed=seed)
    training_targets = dataset.targets[splits.train]
    validate_training_targets(training_targets)
    model = build_pipeline(seed=seed)
    # Balanced weights are calculated independently by each estimator using
    # only its training targets. Validation and test rows never enter fit().
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(dataset.texts[splits.train], training_targets)
    model.toxic_label_order_ = list(LABELS)
    _validate_model(model)
    validation_targets = dataset.targets[splits.validation]
    validation_texts = dataset.texts[splits.validation]
    original_validation_probabilities = model.predict_proba(validation_texts)
    _validate_arrays(validation_targets, original_validation_probabilities)
    model_path = destination / "model.joblib"
    with model_path.open("xb") as stream:
        joblib.dump(model, stream, compress=3)
    # Thresholds and metrics are based exclusively on fresh inference through
    # the reloaded model, rather than arrays from a previous fit/checkpoint.
    exported_model = _validate_model(joblib.load(model_path))
    validation_probabilities = exported_model.predict_proba(validation_texts)
    _validate_arrays(validation_targets, validation_probabilities)
    if not np.array_equal(original_validation_probabilities, validation_probabilities):
        raise RuntimeError("Reloaded model does not reproduce the fitted model's validation probabilities")
    thresholds = select_thresholds(validation_targets, validation_probabilities)
    validation_predictions_path = destination / "private" / "validation_predictions.npz"
    with validation_predictions_path.open("xb") as stream:
        np.savez_compressed(
            stream, row_indices=splits.validation, targets=validation_targets,
            probabilities=validation_probabilities,
        )
    validation_metrics_path = write_json(destination / "validation_metrics.json", {
        "method": METHOD,
        "split": "validation",
        "synthetic": bool(dataset.summary["synthetic"]),
        "metrics_at_0_5": evaluate_predictions(validation_targets, validation_probabilities, np.full(len(LABELS), 0.5)),
        "metrics_at_validation_thresholds": evaluate_predictions(validation_targets, validation_probabilities, thresholds),
        "selection_note": "Thresholds were selected using this validation partition; these metrics are selection diagnostics.",
    })
    positives = training_targets.sum(axis=0)
    negatives = len(training_targets) - positives
    metadata.update({
        "model_family": "word TF-IDF and one-vs-rest logistic regression",
        "synthetic": bool(dataset.summary["synthetic"]),
        "engineering_smoke": bool(dataset.summary["synthetic"]),
        "vocabulary_fit_partition": "train",
        "class_weight_fit_partition": "train",
        "training_class_weights": {
            label: {
                "negative": float(len(training_targets) / (2 * negatives[column])),
                "positive": float(len(training_targets) / (2 * positives[column])),
            }
            for column, label in enumerate(LABELS)
        },
        "threshold_selection": "per-label validation F1; >= comparator; ties nearest 0.5 then larger; missing class keeps 0.5",
        "threshold_grid": list(DEFAULT_THRESHOLD_GRID),
        "export_matches_fitted_validation_predictions": True,
        "fresh_validation_predictions": True,
        "test_status": "unscored during development",
        "protocol_note": PROTOCOL_NOTE,
    })
    freeze_run(
        destination,
        config={
            "method": METHOD, "seed": seed, "label_order": list(LABELS),
            "conflict_policy": conflict_policy, "empty_text_policy": empty_text_policy,
            "tfidf": {
                "analyzer": "word", "ngram_range": [1, 2], "min_df": 2,
                "max_features": 50_000, "sublinear_tf": True, "lowercase": True,
                "norm": "l2", "dtype": "float64",
            },
            "classifier": {
                "wrapper": "OneVsRestClassifier", "solver": "liblinear", "C": 4.0,
                "class_weight": "balanced", "max_iter": 1000, "tol": 1e-4,
                "random_state": seed, "n_jobs": 1,
            },
        },
        data_summary={**dataset.summary, "splits": split_summary(dataset, splits)},
        source_path=source_snapshot, model_path=model_path, thresholds=thresholds,
        splits=splits, metadata=metadata,
        extra_artifacts=[validation_predictions_path, validation_metrics_path],
    )
    return {
        "run_dir": str(destination), "stage": "developed_and_frozen",
        "synthetic": bool(dataset.summary["synthetic"]), "label_order": list(LABELS),
        "checks": {
            "export_matches_fitted_validation_predictions": True,
            "fresh_validation_predictions": True,
            "test_unscored_during_development": True,
        },
    }


def evaluate_run(run_dir: str | Path, *, data_path: str | Path | None = None) -> dict[str, Any]:
    """Evaluate frozen thresholds and model once on the internal test split."""
    destination = Path(run_dir).resolve()
    manifest = verify_frozen_run(destination, source_path=data_path)
    config = read_json(destination / "config.json")
    metadata = read_json(destination / "metadata.json")
    if config.get("method") != METHOD or config.get("label_order") != list(LABELS):
        raise ValueError("Run is not a compatible frozen TF-IDF/logistic-regression experiment")
    seed = _validate_seed(config["seed"])
    source = Path(data_path).resolve() if data_path is not None else resolve_run_path(destination, manifest["source_path"])
    dataset = load_dataset(
        source, conflict_policy=config["conflict_policy"], empty_text_policy=config["empty_text_policy"],
    )
    splits = load_splits(destination)
    expected_splits = split_dataset(dataset, seed=seed)
    if any(
        not np.array_equal(getattr(splits, name), getattr(expected_splits, name))
        for name in ("train", "validation", "test")
    ):
        raise ValueError("Frozen splits do not match the verified data and recorded seed")
    model = _validate_model(joblib.load(resolve_run_path(destination, manifest["model_path"])))
    thresholds = load_thresholds(destination)
    # The claim is exclusive and persistent even if subsequent inference fails.
    # It is acquired before reading test targets or computing test predictions.
    begin_test_evaluation(destination, source_path=source)
    targets = dataset.targets[splits.test]
    probabilities = model.predict_proba(dataset.texts[splits.test])
    results = {
        "method": METHOD,
        "synthetic": bool(dataset.summary["synthetic"]),
        "engineering_smoke": bool(metadata["engineering_smoke"]),
        "split": "internal exploratory test",
        "protocol_note": metadata["protocol_note"],
        "selection_policy": "fixed reference settings; thresholds frozen from validation only",
        "metrics_at_0_5": evaluate_predictions(targets, probabilities, np.full(len(LABELS), 0.5)),
        "metrics_at_frozen_validation_thresholds": evaluate_predictions(targets, probabilities, thresholds),
    }
    complete_test_evaluation(
        destination, public_results=results, targets=targets,
        probabilities=probabilities, row_indices=splits.test, source_path=source,
    )
    return results


def run_smoke(run_dir: str | Path, *, rows: int = 240, seed: int = 42) -> dict[str, Any]:
    """Exercise train/save/reload/freeze/evaluate offline on synthetic text."""
    seed = _validate_seed(seed)
    with tempfile.TemporaryDirectory(prefix="toxic-baseline-fixture-") as temporary:
        source = create_synthetic_fixture(Path(temporary) / "synthetic.csv", rows=rows, seed=seed)
        developed = run_development(source, run_dir, seed=seed, quiet=True)
    results = evaluate_run(run_dir)
    destination = Path(run_dir).resolve()
    with np.load(destination / "private" / "test_predictions.npz", allow_pickle=False) as predictions:
        finite = bool(np.isfinite(predictions["probabilities"]).all())
        six_columns = predictions["probabilities"].shape[1] == len(LABELS)
    if not finite or not six_columns or not results["engineering_smoke"]:
        raise RuntimeError("Offline baseline smoke assertions failed")
    return {
        "status": "passed", "run_dir": str(destination), "synthetic": True,
        "purpose": "engineering mechanics only; no real-data performance claim",
        "checks": {
            **developed["checks"], "train_only_tfidf_and_balanced_weights": True,
            "finite_test_probabilities": finite, "explicit_six_label_order": six_columns,
            "frozen_test_evaluation_completed": True,
        },
    }


# Concise programmatic names are kept alongside the CLI-aligned names.
develop = run_development
evaluate = evaluate_run


def _parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    stages = parser.add_subparsers(dest="stage", required=True)
    development = stages.add_parser("develop", help="Fit train/validation reference pipeline and freeze a new run")
    development.add_argument("--data", required=True, type=Path)
    development.add_argument("--run-dir", required=True, type=Path)
    development.add_argument("--seed", default=42, type=int)
    development.add_argument("--conflict-policy", choices=("error", "keep"), default="error")
    development.add_argument("--empty-text-policy", choices=("error", "drop"), default="error")
    evaluation = stages.add_parser("evaluate", help="Evaluate a frozen reference run once on its internal test split")
    evaluation.add_argument("--run-dir", required=True, type=Path)
    evaluation.add_argument("--data", type=Path, help="Optional byte-identical replacement for the frozen source CSV")
    smoke = stages.add_parser("smoke", help="Offline synthetic engineering mechanics check; not a benchmark")
    smoke.add_argument("--run-dir", required=True, type=Path)
    smoke.add_argument("--rows", default=240, type=int)
    smoke.add_argument("--seed", default=42, type=int)
    return parser


def main(argv: list[str] | None = None) -> int:
    parser = _parser()
    arguments = parser.parse_args(argv)
    try:
        if arguments.stage == "develop":
            result = run_development(
                arguments.data, arguments.run_dir, seed=arguments.seed,
                conflict_policy=arguments.conflict_policy, empty_text_policy=arguments.empty_text_policy,
            )
        elif arguments.stage == "evaluate":
            result = evaluate_run(arguments.run_dir, data_path=arguments.data)
        else:
            result = run_smoke(arguments.run_dir, rows=arguments.rows, seed=arguments.seed)
    except (ValueError, FileNotFoundError, FileExistsError, RuntimeError, ConvergenceWarning) as error:
        parser.exit(2, f"error: {error}\n")
    print(json.dumps(result, indent=2, sort_keys=True, allow_nan=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/toxic-comment-classification/src/toxic_comments/data.py
"""Strict CSV loading and seeded splits of normalized exact-text groups.

Grouping catches exact text after Unicode/case/whitespace normalization. It does
not detect paraphrases or other near duplicates. Splitting is not stratified.
"""

from __future__ import annotations

import csv
import hashlib
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

LABELS = ("identity_hate", "insult", "obscene", "severe_toxic", "threat", "toxic")
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)


def file_sha256(path: str | Path) -> str:
    """Hash bytes without loading an entire dataset into memory."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_text(text: str) -> str:
    """Normalize only for grouping; model inputs retain their original text."""
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())


@dataclass(frozen=True)
class LoadedDataset:
    frame: pd.DataFrame
    groups: np.ndarray
    source_sha256: str
    summary: dict[str, Any]

    @property
    def texts(self) -> np.ndarray:
        return self.frame["comment_text"].to_numpy(dtype=str)

    @property
    def targets(self) -> np.ndarray:
        return self.frame.loc[:, list(LABELS)].to_numpy(dtype=np.uint8)


@dataclass(frozen=True)
class SplitIndices:
    train: np.ndarray
    validation: np.ndarray
    test: np.ndarray

    def as_dict(self) -> dict[str, np.ndarray]:
        return {name: getattr(self, name) for name in ("train", "validation", "test")}


def load_dataset(
    path: str | Path,
    *,
    conflict_policy: str = "error",
    empty_text_policy: str = "error",
) -> LoadedDataset:
    """Load official train.csv or the historical cleaned mirror's train rows.

    ``keep`` retains differently labeled normalized duplicates as separate rows
    in the same group. The default ``error`` requires the caller to resolve or
    explicitly accept conflicting annotations. ``empty_text_policy='drop'``
    excludes only rows whose normalized text is empty; its row count is reported
    separately from the mirror's excluded nontraining rows. IDs and extra
    columns, including the label-derived ``toxicity`` field, are not model inputs.
    """
    if conflict_policy not in {"error", "keep"}:
        raise ValueError("conflict_policy must be 'error' or 'keep'")
    if empty_text_policy not in {"error", "drop"}:
        raise ValueError("empty_text_policy must be 'error' or 'drop'")
    source = Path(path).resolve()
    source_hash = file_sha256(source)
    with source.open("r", encoding="utf-8-sig", newline="") as stream:
        header = next(csv.reader(stream), [])
    if not header or len(set(header)) != len(header):
        raise ValueError("CSV must have a header with unique column names")
    # Do not infer numeric-looking comments as numbers, or treat 'NA' as missing.
    frame = pd.read_csv(source, dtype={"comment_text": str, "id": str}, keep_default_na=False)
    input_rows = len(frame)
    if "set" in frame.columns:
        frame = frame.loc[frame["set"] == "train"].copy()
        source_format = "preprocessed_mirror_schema"
    else:
        source_format = "jigsaw_train_schema"
    excluded_nontrain_rows = input_rows - len(frame)
    if frame.empty:
        raise ValueError("Dataset has no labeled training rows")
    if "severe_tox" in frame.columns:
        if "severe_toxic" in frame.columns:
            canonical = pd.to_numeric(frame["severe_toxic"], errors="coerce")
            alias = pd.to_numeric(frame["severe_tox"], errors="coerce")
            if not np.array_equal(canonical.to_numpy(), alias.to_numpy(), equal_nan=True):
                raise ValueError("severe_tox and severe_toxic columns disagree")
        else:
            frame = frame.rename(columns={"severe_tox": "severe_toxic"})
    missing = [column for column in ("comment_text", *LABELS) if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(missing)}")
    texts = frame["comment_text"]
    if texts.isna().any() or not texts.map(lambda value: isinstance(value, str)).all():
        raise ValueError("comment_text must contain nonmissing strings")
    normalized = texts.map(normalize_text)
    empty = normalized.eq("")
    excluded_empty_text_rows = int(empty.sum())
    if excluded_empty_text_rows:
        if empty_text_policy == "error":
            raise ValueError(
                "comment_text must not contain empty or whitespace-only text; "
                "explicitly use empty_text_policy='drop' to exclude these rows"
            )
        frame = frame.loc[~empty].copy()
        normalized = normalized.loc[~empty]
        if frame.empty:
            raise ValueError("Dataset has no nonempty labeled training rows after dropping empty text")
    missing_ids = duplicate_ids = duplicate_id_groups = None
    if "id" in frame.columns:
        identifiers = frame["id"].fillna("").astype(str).str.strip()
        missing_ids = int(identifiers.eq("").sum())
        present = identifiers.ne("")
        identifier_texts = pd.DataFrame({"identifier": identifiers[present], "text": normalized[present]})
        if identifier_texts.groupby("identifier")["text"].nunique().gt(1).any():
            raise ValueError("A repeated nonblank id refers to different normalized texts")
        identifier_sizes = identifiers[present].value_counts()
        duplicate_ids = int((identifier_sizes - 1).clip(lower=0).sum())
        duplicate_id_groups = int(identifier_sizes.gt(1).sum())
    for label in LABELS:
        values = pd.to_numeric(frame[label], errors="coerce")
        if values.isna().any() or not values.isin([0, 1]).all():
            raise ValueError(f"Label {label} must contain only nonmissing binary 0/1 values")
        frame[label] = values.astype(np.uint8)
    # Hashes stay private. Sorting group hashes later makes assignment independent
    # of row ordering, while every row and original annotation is retained.
    groups = normalized.map(lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest())
    annotations = pd.DataFrame({"group": groups})
    for label in LABELS:
        annotations[label] = frame[label]
    label_variation = annotations.groupby("group")[list(LABELS)].nunique()
    conflict_count = int(label_variation.gt(1).any(axis=1).sum())
    if conflict_count and conflict_policy == "error":
        raise ValueError(
            f"Found {conflict_count} normalized-text groups with conflicting labels; "
            "resolve annotations or explicitly use conflict_policy='keep'"
        )
    sizes = groups.value_counts()
    synthetic = False
    if "_synthetic_fixture" in frame.columns:
        marker = pd.to_numeric(frame["_synthetic_fixture"], errors="coerce")
        if not marker.eq(1).all():
            raise ValueError("_synthetic_fixture must equal 1 for every retained row")
        synthetic = True
    clean = frame.loc[:, ["comment_text", *LABELS]].reset_index(drop=True)
    summary = {
        "source_format": source_format,
        "source_provenance": "synthetic_engineering_fixture" if synthetic else "user_supplied_origin_unverified",
        "synthetic": synthetic,
        "input_rows": input_rows,
        "excluded_nontrain_rows": excluded_nontrain_rows,
        "excluded_empty_text_rows": excluded_empty_text_rows,
        "retained_rows": len(clean),
        "source_id_column_present": "id" in frame.columns,
        "missing_source_ids": missing_ids,
        "duplicate_source_ids": duplicate_ids,
        "duplicate_source_id_groups": duplicate_id_groups,
        "label_order": list(LABELS),
        "positive_counts": {label: int(clean[label].sum()) for label in LABELS},
        "normalized_text_groups": int(len(sizes)),
        "duplicate_groups": int(sizes.gt(1).sum()),
        "rows_in_duplicate_groups": int(sizes.loc[sizes.gt(1)].sum()),
        "conflicting_label_groups": conflict_count,
        "conflict_policy": conflict_policy,
        "empty_text_policy": empty_text_policy,
        "grouping": "NFKC, casefold, Unicode whitespace collapse; exact matches only",
        "near_duplicate_detection": False,
    }
    if file_sha256(source) != source_hash:
        raise ValueError("Dataset changed while it was being loaded")
    return LoadedDataset(clean, groups.to_numpy(dtype=str), source_hash, summary)


def split_dataset(dataset: LoadedDataset, *, seed: int = 42) -> SplitIndices:
    """Seeded 70/15/15 allocation by group count, with nonempty partitions.

    Row fractions can differ when groups have different sizes. No label
    stratification is attempted; downstream metrics expose missing coverage.
    """
    if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    unique = np.unique(dataset.groups)
    if len(unique) < 3:
        raise ValueError("At least three normalized-text groups are required for train/validation/test")
    shuffled = np.random.default_rng(seed).permutation(unique)
    n_train = max(1, min(len(unique) - 2, int(len(unique) * SPLIT_FRACTIONS[0])))
    n_validation = max(1, min(len(unique) - n_train - 1, int(len(unique) * SPLIT_FRACTIONS[1])))
    group_sets = (
        shuffled[:n_train],
        shuffled[n_train : n_train + n_validation],
        shuffled[n_train + n_validation :],
    )
    arrays = [np.flatnonzero(np.isin(dataset.groups, group_set)) for group_set in group_sets]
    return SplitIndices(*arrays)


def split_summary(dataset: LoadedDataset, splits: SplitIndices) -> dict[str, Any]:
    return {
        "strategy": "seeded normalized exact-text group shuffle; not stratified",
        "target_group_fractions": dict(zip(("train", "validation", "test"), SPLIT_FRACTIONS)),
        "partitions": {
            name: {
                "rows": len(indices),
                "row_fraction": len(indices) / len(dataset.frame),
                "groups": int(len(np.unique(dataset.groups[indices]))),
                "positive_counts": {
                    label: int(dataset.targets[indices, column].sum())
                    for column, label in enumerate(LABELS)
                },
            }
            for name, indices in splits.as_dict().items()
        },
    }


def create_synthetic_fixture(path: str | Path, *, rows: int = 240, seed: int = 42) -> Path:
    """Write an easy, deterministic engineering fixture, never benchmark data."""
    if not 60 <= rows <= 10000:
        raise ValueError("Synthetic fixture rows must be between 60 and 10000")
    rng = np.random.default_rng(seed)
    markers = ("identitymarker", "insultmarker", "obscenemarker", "severemarker", "threatmarker", "toxicmarker")
    targets = rng.integers(0, 2, size=(rows, len(LABELS)), dtype=np.uint8)
    texts = [
        "Synthetic fixture comment " + str(row) + " friendly conversation "
        + " ".join(markers[column] for column in range(len(LABELS)) if targets[row, column])
        for row in range(rows)
    ]
    frame = pd.DataFrame(targets, columns=LABELS)
    frame.insert(0, "comment_text", texts)
    frame["_synthetic_fixture"] = 1
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(destination, index=False)
    return destination


In [ ]:
%%writefile /content/toxic-comment-classification/src/toxic_comments/metrics.py
"""Average precision, coverage-aware evaluation, and validation-only thresholds."""

from __future__ import annotations

from typing import Any, Sequence

import numpy as np
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

from .data import LABELS

DEFAULT_THRESHOLD_GRID = tuple(value / 20 for value in range(1, 20))


def _validate_arrays(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    targets = np.asarray(y_true)
    scores = np.asarray(probabilities, dtype=float)
    if targets.ndim != 2 or targets.shape != scores.shape or targets.shape[1] != len(LABELS):
        raise ValueError(f"Targets and probabilities must have the same (n, {len(LABELS)}) shape")
    if targets.shape[0] == 0:
        raise ValueError("Cannot evaluate an empty partition")
    if not np.isin(targets, [0, 1]).all():
        raise ValueError("Targets must be binary 0/1 values")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and within [0, 1]")
    return targets.astype(np.uint8), scores


def _validate_thresholds(thresholds: Sequence[float]) -> np.ndarray:
    values = np.asarray(thresholds, dtype=float)
    if values.shape != (len(LABELS),) or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError(f"Thresholds must be {len(LABELS)} finite values within [0, 1]")
    return values


def select_thresholds(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    *,
    grid: Sequence[float] = DEFAULT_THRESHOLD_GRID,
) -> np.ndarray:
    """Maximize each label's validation F1 on a fixed, bounded threshold grid.

    Ties prefer the value nearest 0.5, then the larger value. A label missing
    either validation class keeps 0.5 because its threshold cannot be selected
    meaningfully from that partition. Callers must pass validation data only.
    """
    targets, scores = _validate_arrays(y_true, probabilities)
    candidates = np.asarray(grid, dtype=float)
    if candidates.ndim != 1 or not 1 <= len(candidates) <= 101:
        raise ValueError("Threshold grid must contain between 1 and 101 values")
    if not np.isfinite(candidates).all() or ((candidates <= 0) | (candidates >= 1)).any():
        raise ValueError("Threshold grid values must be finite and strictly between 0 and 1")
    if len(np.unique(candidates)) != len(candidates):
        raise ValueError("Threshold grid values must be unique")
    selected = np.full(len(LABELS), 0.5, dtype=float)
    for column in range(len(LABELS)):
        target = targets[:, column]
        if len(np.unique(target)) != 2:
            continue
        scored = [
            (float(f1_score(target, scores[:, column] >= threshold, zero_division=0)), float(threshold))
            for threshold in candidates
        ]
        best_score = max(value for value, _ in scored)
        tied = [threshold for value, threshold in scored if np.isclose(value, best_score, rtol=0, atol=1e-12)]
        # Round the distance so representational noise in 0.45/0.55 cannot
        # defeat the documented higher-threshold tie breaker.
        selected[column] = min(tied, key=lambda threshold: (round(abs(threshold - 0.5), 12), -threshold))
    return selected


def evaluate_predictions(
    y_true: np.ndarray, probabilities: np.ndarray, thresholds: Sequence[float]
) -> dict[str, Any]:
    """Return aggregates safe to share without rows, comment text, or IDs.

    Average precision is sklearn's noninterpolated average_precision_score,
    never a trapezoidal PR integral. Single-class AP/ROC AUC are null: an
    all-positive AP of 1 would convey no ranking information. The six-label
    macro scores are null unless every label is eligible; separately named
    diagnostics average the eligible subset and expose its denominator. F1
    needs positive support but not negative support.
    """
    targets, scores = _validate_arrays(y_true, probabilities)
    threshold_values = _validate_thresholds(thresholds)
    predicted = scores >= threshold_values
    per_label: dict[str, dict[str, Any]] = {}
    ranking_labels: list[str] = []
    f1_labels: list[str] = []
    for column, label in enumerate(LABELS):
        target, prediction = targets[:, column], predicted[:, column]
        positives = int(target.sum())
        negatives = int(len(target) - positives)
        true_positive = int(np.logical_and(target == 1, prediction).sum())
        false_positive = int(np.logical_and(target == 0, prediction).sum())
        false_negative = positives - true_positive
        true_negative = negatives - false_positive
        both_classes = positives > 0 and negatives > 0
        has_positive = positives > 0
        predicted_positive = true_positive + false_positive
        per_label[label] = {
            "positive_support": positives,
            "negative_support": negatives,
            "predicted_positive": predicted_positive,
            "average_precision": float(average_precision_score(target, scores[:, column])) if both_classes else None,
            "roc_auc": float(roc_auc_score(target, scores[:, column])) if both_classes else None,
            "f1": float(f1_score(target, prediction, zero_division=0)) if has_positive else None,
            "precision": true_positive / predicted_positive if predicted_positive else None,
            "recall": true_positive / positives if has_positive else None,
            "threshold": float(threshold_values[column]),
            "ranking_status": "defined" if both_classes else ("no_positives" if not positives else "no_negatives"),
            "f1_status": "defined" if has_positive else "no_positives",
            "confusion": {
                "true_positive": true_positive,
                "false_positive": false_positive,
                "false_negative": false_negative,
                "true_negative": true_negative,
            },
        }
        if both_classes:
            ranking_labels.append(label)
        if has_positive:
            f1_labels.append(label)

    def macro(metric: str, eligible: list[str]) -> float | None:
        return float(np.mean([per_label[label][metric] for label in eligible])) if eligible else None

    flattened = targets.ravel()
    micro_has_positive = bool(flattened.any())
    micro_both_classes = micro_has_positive and bool((flattened == 0).any())
    return {
        "primary_metric": "macro_average_precision",
        "rows": len(targets),
        "macro_average_precision": macro("average_precision", ranking_labels) if len(ranking_labels) == len(LABELS) else None,
        "macro_average_precision_eligible_labels": macro("average_precision", ranking_labels),
        "macro_roc_auc": macro("roc_auc", ranking_labels) if len(ranking_labels) == len(LABELS) else None,
        "macro_roc_auc_eligible_labels": macro("roc_auc", ranking_labels),
        "macro_f1": macro("f1", f1_labels) if len(f1_labels) == len(LABELS) else None,
        "macro_f1_eligible_labels": macro("f1", f1_labels),
        "micro_f1": float(f1_score(targets, predicted, average="micro", zero_division=0)) if micro_has_positive else None,
        "micro_average_precision": float(average_precision_score(flattened, scores.ravel())) if micro_both_classes else None,
        "coverage": {
            "total_labels": len(LABELS),
            "ranking_eligible_count": len(ranking_labels),
            "ranking_eligible_labels": ranking_labels,
            "f1_eligible_count": len(f1_labels),
            "f1_eligible_labels": f1_labels,
            "macro_averaging": "six-label macro scores require full coverage; separately named eligible-label diagnostics use their stated subset",
        },
        "per_label": per_label,
    }


In [ ]:
%%writefile /content/toxic-comment-classification/src/toxic_comments/neural.py
"""Repaired original random-initialized RoBERTa experiment.

The real-data path loads a ``roberta-base`` tokenizer, never pretrained model
weights. Neural imports are lazy, so the core pipeline and CLI help work without
the optional neural environment. The smoke path builds its tokenizer locally.
"""

from __future__ import annotations

import argparse
import importlib.metadata
import itertools
import json
import math
import os
import random
import shutil
import tempfile
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np

from .data import LABELS, create_synthetic_fixture, load_dataset, split_dataset, split_summary
from .metrics import DEFAULT_THRESHOLD_GRID, evaluate_predictions, select_thresholds

ORIGINAL_ARCHITECTURE = {
    "vocab_size": 50265,
    "hidden_size": 512,
    "num_hidden_layers": 8,
    "num_attention_heads": 8,
    "intermediate_size": 2048,
    "max_position_embeddings": 512,
    "hidden_dropout_prob": 0.1,
    "attention_probs_dropout_prob": 0.1,
}
SMOKE_ARCHITECTURE = {
    "hidden_size": 32,
    "num_hidden_layers": 1,
    "num_attention_heads": 4,
    "intermediate_size": 64,
    "max_position_embeddings": 64,
    "hidden_dropout_prob": 0.1,
    "attention_probs_dropout_prob": 0.1,
}


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 42
    epochs: int = 5
    patience: int = 3
    batch_size: int = 8
    accumulation_steps: int = 32
    learning_rate: float = 4e-5
    weight_decay: float = 1e-2
    warmup_fraction: float = 0.1
    max_length: int = 256
    max_gradient_norm: float = 1.0
    num_workers: int = 0
    cpu_threads: int = 4
    device: str = "cpu"
    amp: bool = True
    tokenizer_name_or_path: str = "roberta-base"
    tokenizer_revision: str | None = None
    local_files_only: bool = False
    conflict_policy: str = "error"
    empty_text_policy: str = "error"

    def __post_init__(self) -> None:
        for name in ("epochs", "patience", "batch_size", "accumulation_steps", "cpu_threads"):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, int) or value < 1:
                raise ValueError(f"{name} must be a positive integer")
        if isinstance(self.seed, bool) or not isinstance(self.seed, int) or not 0 <= self.seed < 2**32:
            raise ValueError("seed must be an integer in [0, 2**32)")
        if isinstance(self.num_workers, bool) or not isinstance(self.num_workers, int) or self.num_workers < 0:
            raise ValueError("num_workers must be a nonnegative integer")
        if isinstance(self.max_length, bool) or not isinstance(self.max_length, int) or not 4 <= self.max_length <= 510:
            raise ValueError("max_length must be an integer between 4 and 510")
        for name in ("learning_rate", "max_gradient_norm"):
            if not math.isfinite(getattr(self, name)) or getattr(self, name) <= 0:
                raise ValueError(f"{name} must be finite and positive")
        if not math.isfinite(self.weight_decay) or self.weight_decay < 0:
            raise ValueError("weight_decay must be finite and nonnegative")
        if not math.isfinite(self.warmup_fraction) or not 0 <= self.warmup_fraction < 1:
            raise ValueError("warmup_fraction must be within [0, 1)")
        if self.device not in {"cpu", "cuda"}:
            raise ValueError("device must be 'cpu' or 'cuda'; GPU use is explicit")
        if self.conflict_policy not in {"error", "keep"}:
            raise ValueError("conflict_policy must be 'error' or 'keep'")
        if self.empty_text_policy not in {"error", "drop"}:
            raise ValueError("empty_text_policy must be 'error' or 'drop'")


def _dependencies() -> SimpleNamespace:
    # Configure cuBLAS before any CUDA context is initialized. Determinism still
    # depends on using the same software and hardware; cross-platform equality
    # is not promised.
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    try:
        import torch
        import transformers
        from transformers import (
            AutoTokenizer,
            PreTrainedTokenizerFast,
            RobertaConfig,
            RobertaForSequenceClassification,
            get_cosine_schedule_with_warmup,
        )
    except ImportError as error:
        raise ImportError(
            "Neural dependencies are unavailable. Install requirements-neural.txt "
            "in the documented Python environment, then run scripts/check_neural_environment.py."
        ) from error
    return SimpleNamespace(
        torch=torch,
        transformers=transformers,
        AutoTokenizer=AutoTokenizer,
        PreTrainedTokenizerFast=PreTrainedTokenizerFast,
        RobertaConfig=RobertaConfig,
        RobertaForSequenceClassification=RobertaForSequenceClassification,
        get_cosine_schedule_with_warmup=get_cosine_schedule_with_warmup,
    )


def _set_determinism(dependencies: SimpleNamespace, config: TrainConfig) -> Any:
    torch = dependencies.torch
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    torch.set_num_threads(config.cpu_threads)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    if config.device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is unavailable; select --device cpu")
    return torch.device(config.device)


def _seed_worker(worker_id: int) -> None:
    import torch

    seed = torch.initial_seed() % 2**32
    random.seed(seed)
    np.random.seed(seed)


class _TextDataset:
    """Map-style dataset; raw comments never appear in diagnostic output."""

    def __init__(self, texts: np.ndarray, targets: np.ndarray, tokenizer: Any, max_length: int):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, Any]:
        # Import within the worker instead of storing a module on the dataset;
        # datasets must remain pickleable when multiprocessing uses spawn.
        import torch

        encoded = self.tokenizer(
            str(self.texts[index]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {name: encoded[name].squeeze(0) for name in ("input_ids", "attention_mask")}
        item["labels"] = torch.tensor(self.targets[index], dtype=torch.float32)
        return item


def _loader(dataset: Any, indices: np.ndarray, tokenizer: Any, config: TrainConfig, dependencies: Any, *, shuffle: bool) -> Any:
    torch = dependencies.torch
    generator = torch.Generator().manual_seed(config.seed)
    return torch.utils.data.DataLoader(
        _TextDataset(dataset.texts[indices], dataset.targets[indices], tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=shuffle,
        num_workers=config.num_workers,
        pin_memory=config.device == "cuda",
        worker_init_fn=_seed_worker,
        generator=generator,
        drop_last=False,
    )


def validate_training_targets(targets: np.ndarray) -> np.ndarray:
    """Return finite train-only positive weights, rejecting unlearnable labels."""
    labels = np.asarray(targets)
    if labels.ndim != 2 or labels.shape[1] != len(LABELS) or not len(labels):
        raise ValueError(f"Training targets must have nonempty (n, {len(LABELS)}) shape")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Training labels must be finite binary 0/1 values")
    positive = labels.sum(axis=0, dtype=np.float64)
    negative = len(labels) - positive
    missing = [label for column, label in enumerate(LABELS) if not positive[column] or not negative[column]]
    if missing:
        raise ValueError(
            "Every training label needs positive and negative examples for weighted BCE; "
            f"missing coverage: {', '.join(missing)}. Inspect the split without tuning on test results."
        )
    weights = (negative / positive).astype(np.float32)
    if not np.isfinite(weights).all() or (weights <= 0).any():
        raise ValueError("Training positive weights must be finite and positive")
    return weights


def _synthetic_tokenizer(texts: np.ndarray, dependencies: Any, *, max_length: int) -> Any:
    from tokenizers import Tokenizer, models, pre_tokenizers, processors

    special_tokens = ("<s>", "<pad>", "</s>", "<unk>", "<mask>")
    words = sorted({word for text in texts for word in str(text).split()} - set(special_tokens))
    vocabulary = {word: index for index, word in enumerate((*special_tokens, *words))}
    backend = Tokenizer(models.WordLevel(vocabulary, unk_token="<unk>"))
    backend.pre_tokenizer = pre_tokenizers.Whitespace()
    backend.post_processor = processors.TemplateProcessing(
        single="<s> $A </s>",
        special_tokens=[("<s>", vocabulary["<s>"]), ("</s>", vocabulary["</s>"])],
    )
    return dependencies.PreTrainedTokenizerFast(
        tokenizer_object=backend,
        bos_token="<s>",
        pad_token="<pad>",
        eos_token="</s>",
        unk_token="<unk>",
        mask_token="<mask>",
        model_max_length=max_length,
        model_input_names=["input_ids", "attention_mask"],
    )


def _new_model(tokenizer: Any, dependencies: Any, *, smoke: bool) -> Any:
    architecture = dict(SMOKE_ARCHITECTURE if smoke else ORIGINAL_ARCHITECTURE)
    if smoke:
        architecture["vocab_size"] = len(tokenizer)
    elif tokenizer.vocab_size != ORIGINAL_ARCHITECTURE["vocab_size"] or len(tokenizer) != tokenizer.vocab_size:
        raise ValueError("The original experiment requires the unmodified 50,265-token roberta-base tokenizer")
    if not smoke and (tokenizer.bos_token_id, tokenizer.pad_token_id, tokenizer.eos_token_id, tokenizer.unk_token_id) != (0, 1, 2, 3):
        raise ValueError("Tokenizer special-token IDs differ from the original roberta-base contract")
    config = dependencies.RobertaConfig(
        **architecture,
        num_labels=len(LABELS),
        id2label=dict(enumerate(LABELS)),
        label2id={label: index for index, label in enumerate(LABELS)},
        problem_type="multi_label_classification",
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        # Eager attention avoids hardware-dependent fused attention choices.
        attn_implementation="eager",
    )
    # This constructor initializes random weights. No model.from_pretrained
    # call is permitted here: only a saved local export is reloaded below.
    return dependencies.RobertaForSequenceClassification(config)


def _check_label_mapping(model: Any) -> None:
    if model.config.num_labels != len(LABELS):
        raise ValueError("Saved model has an incompatible number of labels")
    if model.config.id2label != dict(enumerate(LABELS)) or model.config.label2id != {label: index for index, label in enumerate(LABELS)}:
        raise ValueError("Saved model label order differs from the explicit six-label contract")


def _autocast(torch: Any, *, enabled: bool) -> Any:
    return torch.amp.autocast("cuda", dtype=torch.float16) if enabled else nullcontext()


def _device_batch(batch: dict[str, Any], device: Any, model: Any) -> dict[str, Any]:
    values = {name: tensor.to(device, non_blocking=device.type == "cuda") for name, tensor in batch.items()}
    input_ids = values["input_ids"]
    if input_ids.min().item() < 0 or input_ids.max().item() >= model.config.vocab_size:
        raise ValueError("Tokenizer produced an input ID outside the model vocabulary")
    return values


def _predict(model: Any, loader: Any, dependencies: Any, device: Any, *, criterion: Any = None, amp_enabled: bool = False) -> tuple[float | None, np.ndarray, np.ndarray]:
    torch = dependencies.torch
    model.eval()
    losses = 0.0
    rows = 0
    probabilities, targets = [], []
    with torch.inference_mode():
        for batch in loader:
            values = _device_batch(batch, device, model)
            with _autocast(torch, enabled=amp_enabled):
                logits = model(input_ids=values["input_ids"], attention_mask=values["attention_mask"]).logits
                loss = criterion(logits, values["labels"]) if criterion is not None else None
            if not torch.isfinite(logits).all().item() or (loss is not None and not torch.isfinite(loss).item()):
                raise FloatingPointError("Nonfinite logits or loss during inference")
            count = len(values["labels"])
            if loss is not None:
                losses += float(loss.item()) * count
            rows += count
            probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())
            targets.append(values["labels"].cpu().numpy().astype(np.uint8))
    if not rows:
        raise ValueError("Cannot predict an empty partition")
    return losses / rows if criterion is not None else None, np.vstack(targets), np.vstack(probabilities)


def _train(model: Any, train_loader: Any, validation_loader: Any, criterion: Any, config: TrainConfig, dependencies: Any, device: Any, checkpoint_path: Path, *, quiet: bool) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    torch = dependencies.torch
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    planned_steps = math.ceil(len(train_loader) / config.accumulation_steps) * config.epochs
    scheduler = dependencies.get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(config.warmup_fraction * planned_steps),
        num_training_steps=planned_steps,
    )
    amp_enabled = config.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    # A public optimizer hook records actual updates. GradScaler can skip a
    # step after overflow; scheduler/step counts must stay unchanged then.
    step_counter = {"value": 0}

    def record_optimizer_step(*_: Any) -> None:
        step_counter["value"] += 1

    step_hook = optimizer.register_step_post_hook(record_optimizer_step)
    history: list[dict[str, Any]] = []
    best = {"epoch": 0, "validation_loss": float("inf"), "optimizer_steps": 0}
    epochs_without_improvement = 0
    try:
        for epoch in range(1, config.epochs + 1):
            model.train()
            training_loss = 0.0
            training_rows = 0
            skipped_steps = 0
            epoch_start_steps = step_counter["value"]
            iterator = iter(train_loader)
            while True:
                micro_batches = list(itertools.islice(iterator, config.accumulation_steps))
                if not micro_batches:
                    break
                group_rows = sum(len(batch["labels"]) for batch in micro_batches)
                optimizer.zero_grad(set_to_none=True)
                for batch in micro_batches:
                    values = _device_batch(batch, device, model)
                    with _autocast(torch, enabled=amp_enabled):
                        logits = model(input_ids=values["input_ids"], attention_mask=values["attention_mask"]).logits
                        loss = criterion(logits, values["labels"])
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError("Nonfinite training loss; no checkpoint/export will be frozen")
                    count = len(values["labels"])
                    # Sample weighting handles both the last partial batch and
                    # a last accumulation group smaller than the configured one.
                    scaler.scale(loss * (count / group_rows)).backward()
                    training_loss += float(loss.item()) * count
                    training_rows += count
                scaler.unscale_(optimizer)
                gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
                if not gradients:
                    raise RuntimeError("Training produced no gradients")
                finite_gradients = torch.stack([torch.isfinite(gradient).all() for gradient in gradients]).all().item()
                if finite_gradients:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_gradient_norm, error_if_nonfinite=True)
                elif not amp_enabled:
                    raise FloatingPointError("Nonfinite training gradients in full precision")
                # Nonfinite AMP gradients were detected by unscale_ and are
                # skipped by GradScaler rather than clipped into NaNs.
                previous_steps = step_counter["value"]
                scaler.step(optimizer)
                scaler.update()
                if step_counter["value"] > previous_steps:
                    scheduler.step()
                else:
                    skipped_steps += 1
            if step_counter["value"] == epoch_start_steps:
                raise FloatingPointError("An epoch completed without any optimizer updates")
            validation_loss, _, _ = _predict(
                model, validation_loader, dependencies, device,
                criterion=criterion, amp_enabled=amp_enabled,
            )
            record = {
                "epoch": epoch,
                "training_loss": training_loss / training_rows,
                "validation_loss": validation_loss,
                "optimizer_steps": step_counter["value"],
                "skipped_amp_steps": skipped_steps,
                "learning_rate": scheduler.get_last_lr()[0],
            }
            history.append(record)
            if not quiet:
                print(f"epoch={epoch} train_loss={record['training_loss']:.6f} validation_loss={validation_loss:.6f} optimizer_steps={step_counter['value']} skipped_amp_steps={skipped_steps}", flush=True)
            if validation_loss < best["validation_loss"]:
                best = {"epoch": epoch, "validation_loss": validation_loss, "optimizer_steps": step_counter["value"]}
                torch.save({**best, "label_order": list(LABELS), "model_state_dict": model.state_dict()}, checkpoint_path)
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= config.patience:
                    break
    finally:
        step_hook.remove()
    return history, best


def _export_best(model: Any, tokenizer: Any, checkpoint_path: Path, model_path: Path, dependencies: Any, device: Any) -> tuple[Any, Any, dict[str, Any]]:
    torch = dependencies.torch
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if checkpoint["label_order"] != list(LABELS):
        raise ValueError("Checkpoint label mapping changed")
    # Restore the selected checkpoint BEFORE save_pretrained. The historical
    # notebook exported last-epoch weights before it restored the best model.
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    _check_label_mapping(model)
    model.save_pretrained(model_path, safe_serialization=True)
    tokenizer.save_pretrained(model_path)
    checkpoint_metadata = {name: checkpoint[name] for name in ("epoch", "validation_loss", "optimizer_steps")}
    del checkpoint
    model.cpu()
    exported = dependencies.RobertaForSequenceClassification.from_pretrained(
        model_path, local_files_only=True, use_safetensors=True, attn_implementation="eager",
    )
    exported_tokenizer = dependencies.AutoTokenizer.from_pretrained(model_path, local_files_only=True, use_fast=True)
    _check_label_mapping(exported)
    for name, tensor in model.state_dict().items():
        if not torch.equal(tensor.cpu(), exported.state_dict()[name].cpu()):
            raise RuntimeError("Saved model parameters differ from the restored best checkpoint")
    if tokenizer.get_vocab() != exported_tokenizer.get_vocab() or tokenizer.special_tokens_map != exported_tokenizer.special_tokens_map:
        raise RuntimeError("Saved tokenizer differs from the tokenizer used in training")
    return exported.to(device).eval(), exported_tokenizer, checkpoint_metadata


def run_development(data_path: str | Path, run_dir: str | Path, *, config: TrainConfig | None = None, quiet: bool = False, _smoke: bool = False) -> dict[str, Any]:
    """Train/select/freeze using train and validation only, never test scores."""
    from .artifacts import create_run_directory, freeze_run, runtime_metadata

    config = config or TrainConfig()
    initial_metadata = runtime_metadata(config.seed)
    dependencies = _dependencies()
    device = _set_determinism(dependencies, config)
    destination = create_run_directory(run_dir)
    private = destination / "private"
    private.mkdir(exist_ok=True)
    source_snapshot = private / "source.csv"
    shutil.copyfile(Path(data_path).resolve(), source_snapshot)
    dataset = load_dataset(source_snapshot, conflict_policy=config.conflict_policy, empty_text_policy=config.empty_text_policy)
    if _smoke and not dataset.summary["synthetic"]:
        raise ValueError("The tiny smoke configuration is restricted to marked synthetic fixtures")
    splits = split_dataset(dataset, seed=config.seed)
    positive_weights = validate_training_targets(dataset.targets[splits.train])
    if _smoke:
        tokenizer = _synthetic_tokenizer(dataset.texts[splits.train], dependencies, max_length=config.max_length)
    else:
        tokenizer = dependencies.AutoTokenizer.from_pretrained(
            config.tokenizer_name_or_path,
            revision=config.tokenizer_revision,
            local_files_only=config.local_files_only,
            use_fast=True,
        )
    model = _new_model(tokenizer, dependencies, smoke=_smoke).to(device)
    if config.max_length + tokenizer.pad_token_id + 1 > model.config.max_position_embeddings:
        raise ValueError("max_length exceeds the configured RoBERTa position capacity")
    torch = dependencies.torch
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weights, dtype=torch.float32, device=device))
    train_loader = _loader(dataset, splits.train, tokenizer, config, dependencies, shuffle=True)
    validation_loader = _loader(dataset, splits.validation, tokenizer, config, dependencies, shuffle=False)
    checkpoint_path = private / "best_checkpoint.pt"
    history, best = _train(model, train_loader, validation_loader, criterion, config, dependencies, device, checkpoint_path, quiet=quiet)
    model_path = destination / "model"
    model, tokenizer, checkpoint_metadata = _export_best(model, tokenizer, checkpoint_path, model_path, dependencies, device)
    # Discard every epoch's cached predictions. Fresh inference through the
    # reloaded export is the sole source for threshold selection and export.
    validation_loader = _loader(dataset, splits.validation, tokenizer, config, dependencies, shuffle=False)
    fresh_loss, validation_targets, validation_probabilities = _predict(
        model, validation_loader, dependencies, device,
        criterion=criterion, amp_enabled=config.amp and device.type == "cuda",
    )
    if not np.array_equal(validation_targets, dataset.targets[splits.validation]):
        raise RuntimeError("Validation inference changed label or row order")
    if not np.isclose(fresh_loss, best["validation_loss"], rtol=1e-5, atol=1e-6):
        raise RuntimeError("Reloaded export does not reproduce the best checkpoint validation loss")
    thresholds = select_thresholds(validation_targets, validation_probabilities)
    validation_predictions_path = private / "validation_predictions.npz"
    np.savez_compressed(validation_predictions_path, targets=validation_targets, probabilities=validation_probabilities, row_indices=splits.validation)
    history_path = destination / "training_history.json"
    history_path.write_text(json.dumps({"epochs": history, "best_checkpoint": checkpoint_metadata, "fresh_validation_loss": fresh_loss}, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    validation_metrics_path = destination / "validation_metrics.json"
    validation_metrics = {
        "split": "validation",
        "synthetic": bool(dataset.summary["synthetic"]),
        "metrics_at_0_5": evaluate_predictions(validation_targets, validation_probabilities, np.full(len(LABELS), 0.5)),
        "metrics_at_validation_thresholds": evaluate_predictions(validation_targets, validation_probabilities, thresholds),
        "selection_note": "Checkpoint and thresholds were selected using this validation partition; these metrics are selection diagnostics.",
    }
    validation_metrics_path.write_text(json.dumps(validation_metrics, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    architecture = {name: getattr(model.config, name) for name in ORIGINAL_ARCHITECTURE}
    metadata = {
        **initial_metadata,
        "model_family": "custom RoBERTa sequence classifier",
        "initialization": "random weights; no pretrained model weights",
        "tokenizer_source": "locally constructed synthetic tokenizer" if _smoke else config.tokenizer_name_or_path,
        "synthetic": bool(dataset.summary["synthetic"]),
        "engineering_smoke": _smoke,
        "checkpoint_selection": "minimum sample-weighted validation BCE loss",
        "best_checkpoint": checkpoint_metadata,
        "train_positive_weights": dict(zip(LABELS, positive_weights.tolist())),
        "threshold_selection": "per-label validation F1; >= comparator; ties nearest 0.5 then larger; missing class keeps 0.5",
        "threshold_grid": list(DEFAULT_THRESHOLD_GRID),
        "deterministic_algorithms": True,
        "cudnn_benchmark": False,
        "attention_implementation": "eager",
        "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "amp_enabled": config.amp and device.type == "cuda",
        "effective_batch_size": config.batch_size * config.accumulation_steps,
        "effective_batch_size_note": "Nominal maximum; the final accumulation group uses its actual sample count.",
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "test_status": "unscored during development",
        "protocol_note": "New internal exploratory holdout; the historical cohort was previously examined, so this is not a pristine external test.",
        "export_matches_best_checkpoint": True,
        "fresh_export_validation_loss_matches_best": True,
    }
    freeze_run(
        destination,
        config={"method": "random_initialized_roberta", "training": asdict(config), "architecture": architecture, "label_order": list(LABELS)},
        data_summary={**dataset.summary, "splits": split_summary(dataset, splits)},
        source_path=source_snapshot,
        model_path=model_path,
        thresholds=thresholds,
        splits=splits,
        metadata=metadata,
        extra_artifacts=[checkpoint_path, validation_predictions_path, history_path, validation_metrics_path],
    )
    return {
        "run_dir": str(destination),
        "stage": "developed_and_frozen",
        "synthetic": bool(dataset.summary["synthetic"]),
        "best_epoch": best["epoch"],
        "label_order": list(LABELS),
        "checks": {"export_matches_best_checkpoint": True, "fresh_validation_predictions": True, "test_unscored_during_development": True},
    }


def evaluate_run(run_dir: str | Path, *, data_path: str | Path | None = None, device: str = "cpu") -> dict[str, Any]:
    """Score a verified frozen run once; marker is acquired before inference."""
    from .artifacts import begin_test_evaluation, complete_test_evaluation, load_splits, load_thresholds, resolve_run_path, verify_frozen_run

    destination = Path(run_dir).resolve()
    manifest = verify_frozen_run(destination, source_path=data_path)
    frozen_config = json.loads((destination / "config.json").read_text(encoding="utf-8"))
    metadata = json.loads((destination / "metadata.json").read_text(encoding="utf-8"))
    if frozen_config["method"] != "random_initialized_roberta" or frozen_config["label_order"] != list(LABELS):
        raise ValueError("Run is not a compatible frozen RoBERTa experiment")
    training_config = dict(frozen_config["training"])
    training_config["device"] = device
    config = TrainConfig(**training_config)
    dependencies = _dependencies()
    torch_device = _set_determinism(dependencies, config)
    source_path = Path(data_path).resolve() if data_path is not None else resolve_run_path(destination, manifest["source_path"])
    dataset = load_dataset(source_path, conflict_policy=config.conflict_policy, empty_text_policy=config.empty_text_policy)
    splits = load_splits(destination)
    expected_splits = split_dataset(dataset, seed=config.seed)
    if any(not np.array_equal(getattr(splits, name), getattr(expected_splits, name)) for name in ("train", "validation", "test")):
        raise ValueError("Frozen splits do not match the verified data and recorded seed")
    model_path = resolve_run_path(destination, manifest["model_path"])
    model = dependencies.RobertaForSequenceClassification.from_pretrained(
        model_path, local_files_only=True, use_safetensors=True, attn_implementation="eager",
    ).to(torch_device).eval()
    tokenizer = dependencies.AutoTokenizer.from_pretrained(model_path, local_files_only=True, use_fast=True)
    _check_label_mapping(model)
    thresholds = load_thresholds(destination)
    begin_test_evaluation(destination, source_path=source_path)
    loader = _loader(dataset, splits.test, tokenizer, config, dependencies, shuffle=False)
    _, targets, probabilities = _predict(
        model, loader, dependencies, torch_device,
        amp_enabled=config.amp and torch_device.type == "cuda",
    )
    if not np.array_equal(targets, dataset.targets[splits.test]):
        raise RuntimeError("Test inference changed label or row order")
    results = {
        "method": "random_initialized_roberta",
        "synthetic": bool(dataset.summary["synthetic"]),
        "engineering_smoke": bool(metadata["engineering_smoke"]),
        "split": "internal exploratory test",
        "protocol_note": metadata["protocol_note"],
        "selection_policy": "best validation loss checkpoint; thresholds frozen from validation only",
        "metrics_at_0_5": evaluate_predictions(targets, probabilities, np.full(len(LABELS), 0.5)),
        "metrics_at_frozen_validation_thresholds": evaluate_predictions(targets, probabilities, thresholds),
    }
    complete_test_evaluation(
        destination, public_results=results, targets=targets,
        probabilities=probabilities, row_indices=splits.test,
        source_path=source_path,
    )
    return results


def run_smoke(run_dir: str | Path) -> dict[str, Any]:
    """Run tiny offline CPU train/save/reload/freeze/evaluate mechanics only."""
    config = TrainConfig(
        epochs=2, patience=2, batch_size=8, accumulation_steps=3,
        learning_rate=1e-3, max_length=32, cpu_threads=1, device="cpu",
        amp=False, local_files_only=True,
    )
    with tempfile.TemporaryDirectory(prefix="toxic-neural-fixture-") as temporary:
        source = create_synthetic_fixture(Path(temporary) / "synthetic.csv", rows=120, seed=config.seed)
        developed = run_development(source, run_dir, config=config, quiet=True, _smoke=True)
    results = evaluate_run(run_dir, device="cpu")
    destination = Path(run_dir).resolve()
    with np.load(destination / "private" / "test_predictions.npz", allow_pickle=False) as predictions:
        finite = bool(np.isfinite(predictions["probabilities"]).all())
        six_columns = predictions["probabilities"].shape[1] == len(LABELS)
    if not finite or not six_columns or not results["engineering_smoke"]:
        raise RuntimeError("Offline neural smoke assertions failed")
    return {
        "status": "passed",
        "run_dir": str(destination),
        "synthetic": True,
        "purpose": "engineering mechanics only; no real-data performance claim",
        "tiny_architecture": SMOKE_ARCHITECTURE,
        "checks": {
            **developed["checks"],
            "local_synthetic_tokenizer": True,
            "cpu_full_precision": True,
            "finite_test_probabilities": finite,
            "explicit_six_label_order": six_columns,
            "frozen_test_evaluation_completed": True,
        },
    }


def check_environment() -> dict[str, Any]:
    """Check the optional environment using a tiny CPU forward pass only."""
    dependencies = _dependencies()
    config = TrainConfig(cpu_threads=1, amp=False)
    device = _set_determinism(dependencies, config)
    tokenizer = _synthetic_tokenizer(np.array(["synthetic environment check"]), dependencies, max_length=16)
    model = _new_model(tokenizer, dependencies, smoke=True).to(device).eval()
    encoded = tokenizer("synthetic environment check", return_tensors="pt")
    with dependencies.torch.inference_mode():
        logits = model(**encoded).logits
    if logits.shape != (1, len(LABELS)) or not dependencies.torch.isfinite(logits).all().item():
        raise RuntimeError("Tiny CPU forward pass failed")
    return {
        "status": "passed",
        "versions": {name: importlib.metadata.version(name) for name in ("torch", "transformers", "tokenizers", "safetensors", "huggingface-hub")},
        "cuda_available": dependencies.torch.cuda.is_available(),
        "checked_device": "cpu",
        "check": "tiny local random model forward pass; no downloads or training",
    }


def resource_diagnostic() -> dict[str, Any]:
    """Count the original architecture on the meta device without weights or data.

    The memory figures cover float32 parameters, gradients and AdamW moments
    only. They exclude activations and temporary tensors, so they cannot be
    used as a measured peak-memory or training-time estimate.
    """
    dependencies = _dependencies()
    config = dependencies.RobertaConfig(
        **ORIGINAL_ARCHITECTURE,
        num_labels=len(LABELS),
        id2label=dict(enumerate(LABELS)),
        label2id={label: index for index, label in enumerate(LABELS)},
        problem_type="multi_label_classification",
        attn_implementation="eager",
    )
    with dependencies.torch.device("meta"):
        model = dependencies.RobertaForSequenceClassification(config)
    _check_label_mapping(model)
    parameters = sum(parameter.numel() for parameter in model.parameters())
    return {
        "architecture": ORIGINAL_ARCHITECTURE,
        "max_length": TrainConfig().max_length,
        "parameter_count": parameters,
        "float32_parameter_memory_mib": parameters * 4 / 1024**2,
        "float32_parameters_gradients_adam_moments_lower_bound_mib": parameters * 16 / 1024**2,
        "memory_note": "Analytical lower bound; excludes activations, temporary tensors and input storage. Peak memory and runtime were not measured.",
        "cuda_available": dependencies.torch.cuda.is_available(),
        "model_initialization": "architecture instantiated on meta device; no full weight allocation or downloads",
        "full_model_training_executed": False,
    }


def _parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    stages = parser.add_subparsers(dest="stage", required=True)
    develop = stages.add_parser("develop", help="Explicitly train/select/freeze the original random RoBERTa on a supplied CSV")
    develop.add_argument("--data", required=True, type=Path)
    develop.add_argument("--run-dir", required=True, type=Path)
    develop.add_argument("--epochs", type=int, default=5)
    develop.add_argument("--patience", type=int, default=3)
    develop.add_argument("--batch-size", type=int, default=8)
    develop.add_argument("--accumulation-steps", type=int, default=32)
    develop.add_argument("--learning-rate", type=float, default=4e-5)
    develop.add_argument("--weight-decay", type=float, default=1e-2)
    develop.add_argument("--max-length", type=int, default=256)
    develop.add_argument("--seed", type=int, default=42)
    develop.add_argument("--device", choices=("cpu", "cuda"), default="cpu")
    develop.add_argument("--num-workers", type=int, default=0)
    develop.add_argument("--cpu-threads", type=int, default=4)
    develop.add_argument("--no-amp", action="store_true", help="Disable CUDA-only mixed precision; CPU always uses float32")
    develop.add_argument("--tokenizer", default="roberta-base", help="roberta-base identifier or equivalent local tokenizer directory; tokenizer only")
    develop.add_argument("--tokenizer-revision", default=None)
    develop.add_argument("--local-files-only", action="store_true")
    develop.add_argument("--conflict-policy", choices=("error", "keep"), default="error")
    develop.add_argument("--empty-text-policy", choices=("error", "drop"), default="error", help="Explicitly drop normalized-empty comments; default rejects them")
    evaluate = stages.add_parser("evaluate", help="Evaluate a verified frozen model once on its internal exploratory test split")
    evaluate.add_argument("--run-dir", required=True, type=Path)
    evaluate.add_argument("--data", type=Path, help="Optional byte-identical replacement for the frozen source CSV")
    evaluate.add_argument("--device", choices=("cpu", "cuda"), default="cpu")
    smoke = stages.add_parser("smoke", help="Offline tiny CPU engineering check with synthetic data; not a benchmark")
    smoke.add_argument("--run-dir", required=True, type=Path)
    stages.add_parser("check-environment", help="Validate neural imports and tiny local CPU forward pass")
    stages.add_parser("diagnose-resources", help="Count the original architecture without training, downloads or full weight allocation")
    return parser


def main(argv: list[str] | None = None) -> int:
    parser = _parser()
    args = parser.parse_args(argv)
    try:
        if args.stage == "develop":
            config = TrainConfig(
                seed=args.seed, epochs=args.epochs, patience=args.patience,
                batch_size=args.batch_size, accumulation_steps=args.accumulation_steps,
                learning_rate=args.learning_rate, weight_decay=args.weight_decay,
                max_length=args.max_length, device=args.device,
                num_workers=args.num_workers, cpu_threads=args.cpu_threads,
                amp=not args.no_amp, tokenizer_name_or_path=args.tokenizer,
                tokenizer_revision=args.tokenizer_revision, local_files_only=args.local_files_only,
                conflict_policy=args.conflict_policy,
                empty_text_policy=args.empty_text_policy,
            )
            result = run_development(args.data, args.run_dir, config=config)
        elif args.stage == "evaluate":
            result = evaluate_run(args.run_dir, data_path=args.data, device=args.device)
        elif args.stage == "smoke":
            result = run_smoke(args.run_dir)
        elif args.stage == "diagnose-resources":
            result = resource_diagnostic()
        else:
            result = check_environment()
    except (ImportError, OSError, ValueError, RuntimeError, FloatingPointError) as error:
        parser.exit(2, f"neural: {error}\n")
    print(json.dumps(result, indent=2, allow_nan=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/toxic-comment-classification/scripts/colab_train.py
"""Resumable Colab runner for the original random eight-layer RoBERTa.

Only the public tokenizer is downloaded. The encoder/classifier is initialized
randomly through toxic_comments.neural._new_model. Private checkpoints and
per-row arrays stay in the chosen run directory; review ZIPs contain aggregates.
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import platform
import random
import shutil
import sys
import tempfile
import time
import zipfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Also support ``python scripts/colab_train.py`` without installing the package.
PROJECT_ROOT = Path(__file__).resolve().parents[1]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np

from toxic_comments import neural as core
from toxic_comments.artifacts import (
    MANIFEST_SCHEMA,
    TEST_MARKER,
    create_run_directory,
    freeze_run,
    read_json,
    runtime_metadata,
    verify_frozen_run,
)
from toxic_comments.data import (
    LABELS,
    create_synthetic_fixture,
    file_sha256,
    load_dataset,
    split_dataset,
    split_summary,
)
from toxic_comments.metrics import DEFAULT_THRESHOLD_GRID, evaluate_predictions, select_thresholds

CHECKPOINT_SCHEMA = 1
LATEST_CHECKPOINT = "private/latest_checkpoint.pt"
BEST_CHECKPOINT = "private/best_checkpoint.pt"
IDENTITY_FILE = "private/run_identity.json"
TOKENIZER_DIRECTORY = "private/source_tokenizer"
FREEZE_TRANSACTION = "private/freeze_transaction.json"
FREEZE_EXTRA_ARTIFACTS = (
    BEST_CHECKPOINT, IDENTITY_FILE, TOKENIZER_DIRECTORY,
    "private/validation_predictions.npz", "training_history.json", "validation_metrics.json",
)
FULL_SOURCE_SHA256 = "bb7a9e1af5282c4c05bf09bf3be3373c4160fe3272a4b26016034ff4bcea5f1e"
FULL_DATA_EXPECTATION = {
    "input_rows": 159571,
    "retained_rows": 159564,
    "excluded_empty_text_rows": 7,
    "conflicting_label_groups": 93,
    "split_rows": {"train": 111731, "validation": 23878, "test": 23955},
}


@dataclass(frozen=True)
class ColabConfig:
    training: core.TrainConfig = field(default_factory=lambda: core.TrainConfig(
        device="cuda", conflict_policy="keep", empty_text_policy="drop",
    ))
    checkpoint_every_updates: int = 100
    gradient_checkpointing: bool = False
    require_full_dataset: bool = True

    def __post_init__(self) -> None:
        if not isinstance(self.training, core.TrainConfig):
            raise ValueError("training must be a TrainConfig")
        if isinstance(self.checkpoint_every_updates, bool) or not isinstance(self.checkpoint_every_updates, int) or self.checkpoint_every_updates < 1:
            raise ValueError("checkpoint_every_updates must be a positive integer")
        for name in ("gradient_checkpointing", "require_full_dataset"):
            if not isinstance(getattr(self, name), bool):
                raise ValueError(f"{name} must be boolean")
        # Worker process RNG/prefetch state is intentionally excluded. Requiring
        # zero workers makes reconstructed iterator offsets fully reviewable.
        if self.training.num_workers != 0:
            raise ValueError("Exact Colab resume requires num_workers=0")


def _canonical_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def _source_code_hashes() -> dict[str, str]:
    paths = sorted((PROJECT_ROOT / "src" / "toxic_comments").glob("*.py"))
    paths.append(Path(__file__).resolve())
    return {path.relative_to(PROJECT_ROOT).as_posix(): file_sha256(path) for path in paths}


def _directory_hashes(path: Path) -> dict[str, str]:
    return {entry.relative_to(path).as_posix(): file_sha256(entry) for entry in sorted(path.rglob("*")) if entry.is_file()}


def _runtime_identity(dependencies: Any) -> dict[str, Any]:
    torch = dependencies.torch
    packages = ("numpy", "pandas", "scipy", "scikit-learn", "joblib", "torch", "transformers", "tokenizers", "safetensors", "huggingface-hub")
    hardware = {
        "cuda_available": torch.cuda.is_available(),
        "cuda_runtime": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "gpus": [
            {"name": torch.cuda.get_device_name(index), "capability": list(torch.cuda.get_device_capability(index)), "total_memory_bytes": torch.cuda.get_device_properties(index).total_memory}
            for index in range(torch.cuda.device_count())
        ] if torch.cuda.is_available() else [],
        "cpu_machine": platform.machine(),
    }
    return {
        "python_version": platform.python_version(),
        "dependencies": {name: importlib.metadata.version(name) for name in packages},
        "hardware": hardware,
    }


def _assert_environment(dependencies: Any, *, smoke: bool, config: ColabConfig) -> None:
    from packaging.version import Version

    if sys.version_info < (3, 12):
        raise RuntimeError("Python 3.12 or newer is required by the pinned, tested core dependencies")
    if Version(dependencies.torch.__version__.split("+")[0]) < Version("2.8.0"):
        raise RuntimeError("torch >=2.8 is required; keep a compatible CUDA-enabled Colab build")
    if not smoke and (config.training.device != "cuda" or not dependencies.torch.cuda.is_available()):
        raise RuntimeError("Full training requires a GPU. In Colab choose Runtime > Change runtime type > GPU, reconnect, and rerun setup")
    if smoke and config.training.device != "cpu":
        raise ValueError("The offline synthetic smoke must use CPU")


def atomic_torch_save(torch: Any, payload: dict[str, Any], path: Path) -> None:
    """Replace a checkpoint only after a complete sibling file is flushed.

    Filesystem durability still depends on the storage provider. A previous
    checkpoint survives a serialization/replace error; incomplete temps are
    removed. Use this only for trusted checkpoints produced by this runner.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)
    temporary = Path(temporary_name)
    try:
        with os.fdopen(descriptor, "wb") as stream:
            torch.save(payload, stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _atomic_json(path: Path, value: Any) -> None:
    descriptor, temporary_name = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)
    temporary = Path(temporary_name)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8") as stream:
            stream.write(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _recover_freeze_checksum(run_dir: Path, identity: dict[str, Any], dependencies: Any) -> None:
    """Finish only an intact, transaction-marked freeze missing its checksum.

    A present checksum is never replaced. Truncated manifests and changed
    artifacts remain fail-closed; this is not a general integrity repair tool.
    """
    manifest_path = run_dir / "manifest.json"
    checksum_path = run_dir / "manifest.sha256"
    transaction_path = run_dir / FREEZE_TRANSACTION
    if not manifest_path.exists() or checksum_path.exists():
        return
    if not transaction_path.is_file() or (run_dir / TEST_MARKER).exists():
        raise ValueError("Incomplete freeze cannot be recovered without its transaction or after test access")
    transaction = read_json(transaction_path)
    expected_transaction = {
        "schema_version": 1,
        "identity_sha256": file_sha256(run_dir / IDENTITY_FILE),
        "latest_checkpoint_sha256": file_sha256(run_dir / LATEST_CHECKPOINT),
        "best_checkpoint_sha256": file_sha256(run_dir / BEST_CHECKPOINT),
    }
    if transaction != expected_transaction:
        raise ValueError("Incomplete freeze transaction or checkpoint identity changed")
    checkpoint = dependencies.torch.load(run_dir / LATEST_CHECKPOINT, map_location="cpu", weights_only=True)
    if (checkpoint.get("schema_version") != CHECKPOINT_SCHEMA
            or checkpoint.get("identity_sha256") != transaction["identity_sha256"]
            or checkpoint.get("label_order") != list(LABELS)
            or checkpoint.get("training_complete") is not True):
        raise ValueError("Incomplete freeze requires the intact completed training checkpoint")
    best_metadata = checkpoint["best"]
    del checkpoint
    best = dependencies.torch.load(run_dir / BEST_CHECKPOINT, map_location="cpu", weights_only=True)
    if best.get("label_order") != list(LABELS) or any(best.get(name) != value for name, value in best_metadata.items()):
        raise ValueError("Incomplete freeze best checkpoint differs from completed training")
    del best
    manifest = read_json(manifest_path)
    if (manifest.get("schema_version") != MANIFEST_SCHEMA
            or manifest.get("label_order") != list(LABELS)
            or manifest.get("source_path") != "private/source.csv"
            or manifest.get("source_sha256") != identity["source_sha256"]
            or manifest.get("model_path") != "model"):
        raise ValueError("Incomplete freeze manifest is incompatible with this Colab run")
    artifacts = ("config.json", "data_summary.json", "metadata.json", "thresholds.json", "private/splits.npz", "model", *FREEZE_EXTRA_ARTIFACTS)
    observed = {}
    for relative in artifacts:
        path = run_dir / relative
        candidates = sorted(path.rglob("*")) if path.is_dir() else [path]
        for candidate in candidates:
            resolved = candidate.resolve()
            if not resolved.is_relative_to(run_dir) or not resolved.is_file():
                raise ValueError("Incomplete freeze contains an invalid artifact path")
            observed[resolved.relative_to(run_dir).as_posix()] = file_sha256(resolved)
    if manifest.get("artifacts_sha256") != observed:
        raise ValueError("Incomplete freeze artifact integrity check failed")
    if not any(name.startswith("model/") for name in observed):
        raise ValueError("Incomplete freeze model export is empty")
    recorded_code = read_json(run_dir / "metadata.json")["code_sha256"]
    expected_code = {name: value for name, value in identity["code_sha256"].items() if name.startswith("src/")}
    if recorded_code != expected_code:
        raise ValueError("Incomplete freeze source code differs from the original run identity")
    descriptor, temporary_name = tempfile.mkstemp(prefix="manifest.sha256.", suffix=".tmp", dir=run_dir)
    temporary = Path(temporary_name)
    try:
        with os.fdopen(descriptor, "w", encoding="ascii") as stream:
            stream.write(file_sha256(manifest_path) + "\n")
            stream.flush()
            os.fsync(stream.fileno())
        if checksum_path.exists():
            raise FileExistsError("A freeze checksum already exists; it cannot be replaced")
        os.replace(temporary, checksum_path)
    finally:
        temporary.unlink(missing_ok=True)
    verify_frozen_run(run_dir)
    transaction_path.unlink()


def capture_rng_state(torch: Any) -> dict[str, Any]:
    """Serialize NumPy RNG as primitives/tensors, safe for weights_only=True."""
    algorithm, keys, position, has_gauss, cached_gaussian = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": {"algorithm": algorithm, "keys": torch.tensor(keys.astype(np.int64)), "position": int(position), "has_gauss": int(has_gauss), "cached_gaussian": float(cached_gaussian)},
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


def restore_rng_state(torch: Any, state: dict[str, Any]) -> None:
    random.setstate(state["python"])
    numpy_state = state["numpy"]
    np.random.set_state((numpy_state["algorithm"], numpy_state["keys"].cpu().numpy().astype(np.uint32), numpy_state["position"], numpy_state["has_gauss"], numpy_state["cached_gaussian"]))
    torch.set_rng_state(state["torch_cpu"].cpu())
    if state["torch_cuda"]:
        if not torch.cuda.is_available() or len(state["torch_cuda"]) != torch.cuda.device_count():
            raise ValueError("Checkpoint CUDA RNG state does not match the current GPU count")
        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])


def _epoch_loader(dataset: Any, indices: np.ndarray, tokenizer: Any, config: core.TrainConfig, dependencies: Any, epoch: int) -> Any:
    """Every epoch has an independent deterministic shuffle and loader RNG."""
    generator = dependencies.torch.Generator().manual_seed(config.seed + epoch)
    return dependencies.torch.utils.data.DataLoader(
        core._TextDataset(dataset.texts[indices], dataset.targets[indices], tokenizer, config.max_length),
        batch_size=config.batch_size, shuffle=True, num_workers=0,
        pin_memory=config.device == "cuda", generator=generator,
        worker_init_fn=core._seed_worker, drop_last=False,
    )


def _optimizer_group(model: Any, micro_batches: list[Any], criterion: Any, optimizer: Any, scaler: Any, scheduler: Any, config: core.TrainConfig, dependencies: Any, device: Any, step_counter: dict[str, int]) -> dict[str, Any]:
    """One accumulation boundary; scale by actual samples, including tails."""
    torch = dependencies.torch
    group_rows = sum(len(batch["labels"]) for batch in micro_batches)
    if not group_rows:
        raise ValueError("An optimizer group must have at least one sample")
    amp_enabled = config.amp and device.type == "cuda"
    optimizer.zero_grad(set_to_none=True)
    numerator = 0.0
    for batch in micro_batches:
        values = core._device_batch(batch, device, model)
        with core._autocast(torch, enabled=amp_enabled):
            logits = model(input_ids=values["input_ids"], attention_mask=values["attention_mask"]).logits
            loss = criterion(logits, values["labels"])
        if not torch.isfinite(loss).item():
            raise FloatingPointError("Nonfinite training loss; this run cannot be frozen")
        rows = len(values["labels"])
        scaler.scale(loss * (rows / group_rows)).backward()
        numerator += float(loss.item()) * rows
    scaler.unscale_(optimizer)
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    if not gradients:
        raise RuntimeError("Training produced no gradients")
    finite = torch.stack([torch.isfinite(gradient).all() for gradient in gradients]).all().item()
    if finite:
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_gradient_norm, error_if_nonfinite=True)
    elif not amp_enabled:
        raise FloatingPointError("Nonfinite training gradients in full precision")
    previous_updates = step_counter["value"]
    scaler.step(optimizer)
    scaler.update()
    updated = step_counter["value"] > previous_updates
    if updated:
        scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    return {"loss_numerator": numerator, "sample_count": group_rows, "skipped_amp_steps": int(not updated)}


def _new_tokenizer(dataset: Any, splits: Any, config: ColabConfig, dependencies: Any, tokenizer_dir: Path, *, smoke: bool) -> tuple[Any, str]:
    training = config.training
    if smoke:
        tokenizer = core._synthetic_tokenizer(dataset.texts[splits.train], dependencies, max_length=training.max_length)
        revision = "local-train-only-synthetic"
    else:
        if training.tokenizer_name_or_path != "roberta-base" or training.local_files_only:
            raise ValueError("Full Colab training downloads the public roberta-base tokenizer once; resume uses its saved local snapshot")
        from huggingface_hub import HfApi

        revision = HfApi().model_info("roberta-base", revision=training.tokenizer_revision or "main").sha
        if not revision:
            raise RuntimeError("Unable to resolve an immutable tokenizer revision")
        tokenizer = dependencies.AutoTokenizer.from_pretrained("roberta-base", revision=revision, use_fast=True)
    tokenizer.save_pretrained(tokenizer_dir)
    return tokenizer, revision


def _validate_identity(identity: dict[str, Any], *, config: ColabConfig, source_path: Path, tokenizer_dir: Path, dependencies: Any, smoke: bool) -> None:
    checks = {
        "configuration": (identity["requested_config"], asdict(config)),
        "source dataset": (identity["source_sha256"], file_sha256(source_path)),
        "source code": (identity["code_sha256"], _source_code_hashes()),
        "tokenizer snapshot": (identity["tokenizer_sha256"], _directory_hashes(tokenizer_dir)),
        "runtime/software/hardware": (identity["runtime"], _runtime_identity(dependencies)),
        "smoke mode": (identity["smoke"], smoke),
    }
    for description, (expected, observed) in checks.items():
        if expected != observed:
            raise ValueError(f"Resume/frozen run {description} changed; use the original setting or a new run directory")


def _verify_full_data(dataset: Any, splits: Any, config: ColabConfig, *, smoke: bool) -> None:
    if smoke:
        if not dataset.summary["synthetic"]:
            raise ValueError("Smoke mode is restricted to marked synthetic data")
        return
    if dataset.summary["synthetic"]:
        raise ValueError("Full training refuses synthetic data")
    if config.require_full_dataset:
        if dataset.source_sha256 != FULL_SOURCE_SHA256:
            raise ValueError("Full source SHA-256 differs from the supplied train_preprocessed.csv. Use the original file bytes; do not silently substitute or resave the dataset")
        for key, expected in FULL_DATA_EXPECTATION.items():
            observed = {name: len(indices) for name, indices in splits.as_dict().items()} if key == "split_rows" else dataset.summary[key]
            if observed != expected:
                raise ValueError(f"Full source expectation failed for {key}: expected {expected}, got {observed}. Do not silently reduce rows; confirm the original source or explicitly start a separate experiment")


def _checkpoint_payload(model: Any, optimizer: Any, scheduler: Any, scaler: Any, dependencies: Any, state: dict[str, Any], identity_hash: str) -> dict[str, Any]:
    return {
        "schema_version": CHECKPOINT_SCHEMA,
        "identity_sha256": identity_hash,
        "label_order": list(LABELS),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "rng_state": capture_rng_state(dependencies.torch),
        **state,
    }


def _train_resumable(model: Any, dataset: Any, splits: Any, tokenizer: Any, criterion: Any, config: ColabConfig, dependencies: Any, device: Any, run_dir: Path, *, resume: bool, quiet: bool, stop_after_steps: int | None = None) -> dict[str, Any]:
    torch = dependencies.torch
    training = config.training
    checkpoint_path = run_dir / LATEST_CHECKPOINT
    identity_hash = file_sha256(run_dir / IDENTITY_FILE)
    microbatches = math.ceil(len(splits.train) / training.batch_size)
    planned_updates = math.ceil(microbatches / training.accumulation_steps) * training.epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=training.learning_rate, weight_decay=training.weight_decay)
    scheduler = dependencies.get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(training.warmup_fraction * planned_updates), num_training_steps=planned_updates)
    amp_enabled = training.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    state = {
        "epoch": 1, "microbatch_offset": 0,
        "training_loss_numerator": 0.0, "training_sample_count": 0,
        "skipped_amp_steps": 0, "epoch_start_optimizer_steps": 0,
        "global_optimizer_steps": 0, "history": [],
        "best": {"epoch": 0, "validation_loss": float("inf"), "optimizer_steps": 0},
        "epochs_without_improvement": 0, "training_complete": False,
    }
    saved_rng = None
    resumed = checkpoint_path.is_file()
    if resumed:
        if not resume:
            raise FileExistsError("An unfinished checkpoint already exists; resume it or choose a new run directory")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if checkpoint.get("schema_version") != CHECKPOINT_SCHEMA or checkpoint.get("identity_sha256") != identity_hash or checkpoint.get("label_order") != list(LABELS):
            raise ValueError("Checkpoint schema, identity or label order is incompatible")
        model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        state = {key: checkpoint[key] for key in state}
        saved_rng = checkpoint["rng_state"]
        del checkpoint
    if state["training_complete"]:
        return {**state, "resumed": resumed, "interrupted": False}
    step_counter = {"value": state["global_optimizer_steps"]}

    def record_update(*_: Any) -> None:
        step_counter["value"] += 1

    step_hook = optimizer.register_step_post_hook(record_update)

    def save_latest() -> None:
        state["global_optimizer_steps"] = step_counter["value"]
        atomic_torch_save(torch, _checkpoint_payload(model, optimizer, scheduler, scaler, dependencies, state, identity_hash), checkpoint_path)

    if not resumed:
        # Even a preemption before the first update can restart from exact state.
        save_latest()
    last_progress = time.monotonic()
    try:
        while state["epoch"] <= training.epochs:
            epoch = state["epoch"]
            loader = _epoch_loader(dataset, splits.train, tokenizer, training, dependencies, epoch)
            offset = state["microbatch_offset"]
            if not 0 <= offset <= len(loader) or (offset != len(loader) and offset % training.accumulation_steps != 0):
                raise ValueError("Checkpoint microbatch offset is not an accumulation boundary")
            iterator = iter(loader)
            for _ in range(offset):
                next(iterator)
            # Reconstructing/skipping a DataLoader must not perturb the global
            # RNG before the next unseen forward pass (dropout in particular).
            if saved_rng is not None:
                restore_rng_state(torch, saved_rng)
                saved_rng = None
            model.train()
            while True:
                micro_batches = list(itertools.islice(iterator, training.accumulation_steps))
                if not micro_batches:
                    break
                previous_updates = step_counter["value"]
                group = _optimizer_group(model, micro_batches, criterion, optimizer, scaler, scheduler, training, dependencies, device, step_counter)
                state["microbatch_offset"] += len(micro_batches)
                state["training_loss_numerator"] += group["loss_numerator"]
                state["training_sample_count"] += group["sample_count"]
                state["skipped_amp_steps"] += group["skipped_amp_steps"]
                state["global_optimizer_steps"] = step_counter["value"]
                actual_update = step_counter["value"] > previous_updates
                interval_due = actual_update and step_counter["value"] % config.checkpoint_every_updates == 0
                stop_due = stop_after_steps is not None and step_counter["value"] >= stop_after_steps
                if interval_due or stop_due:
                    save_latest()
                if not quiet and (interval_due or time.monotonic() - last_progress >= 30):
                    print(f"epoch={epoch}/{training.epochs} samples={state['training_sample_count']}/{len(splits.train)} optimizer_updates={step_counter['value']} loss={state['training_loss_numerator']/state['training_sample_count']:.6f} checkpoint_saved={interval_due or stop_due}", flush=True)
                    last_progress = time.monotonic()
                if stop_due:
                    return {**state, "resumed": resumed, "interrupted": True}
            if step_counter["value"] == state["epoch_start_optimizer_steps"]:
                raise FloatingPointError("An epoch completed without any optimizer updates")
            validation_loader = core._loader(dataset, splits.validation, tokenizer, training, dependencies, shuffle=False)
            validation_loss, _, _ = core._predict(model, validation_loader, dependencies, device, criterion=criterion, amp_enabled=amp_enabled)
            record = {
                "epoch": epoch,
                "training_loss": state["training_loss_numerator"] / state["training_sample_count"],
                "validation_loss": validation_loss,
                "optimizer_steps": step_counter["value"],
                "skipped_amp_steps": state["skipped_amp_steps"],
                "learning_rate": scheduler.get_last_lr()[0],
            }
            state["history"].append(record)
            if validation_loss < state["best"]["validation_loss"]:
                state["best"] = {"epoch": epoch, "validation_loss": validation_loss, "optimizer_steps": step_counter["value"]}
                atomic_torch_save(torch, {**state["best"], "label_order": list(LABELS), "model_state_dict": model.state_dict()}, run_dir / BEST_CHECKPOINT)
                state["epochs_without_improvement"] = 0
            else:
                state["epochs_without_improvement"] += 1
            if not quiet:
                print(f"epoch={epoch} train_loss={record['training_loss']:.6f} validation_loss={validation_loss:.6f} best_epoch={state['best']['epoch']} optimizer_updates={step_counter['value']}", flush=True)
            state.update({
                "epoch": epoch + 1, "microbatch_offset": 0,
                "training_loss_numerator": 0.0, "training_sample_count": 0,
                "skipped_amp_steps": 0, "epoch_start_optimizer_steps": step_counter["value"],
                "training_complete": epoch >= training.epochs or state["epochs_without_improvement"] >= training.patience,
            })
            save_latest()
            if state["training_complete"]:
                break
    finally:
        step_hook.remove()
    return {**state, "resumed": resumed, "interrupted": False}


def _zip_once(path: Path, entries: dict[str, bytes | Path]) -> Path:
    if path.exists():
        return path
    descriptor, temporary_name = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)
    os.close(descriptor)
    temporary = Path(temporary_name)
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for name, value in entries.items():
                if isinstance(value, Path):
                    archive.write(value, arcname=name)
                else:
                    archive.writestr(name, value)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return path


def _json_bytes(value: Any) -> bytes:
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n").encode()


def _review_plots(run_dir: Path) -> list[Path]:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    folder = run_dir / "review_plots"
    folder.mkdir(exist_ok=True)
    history = read_json(run_dir / "training_history.json")["epochs"]
    loss_path = folder / "training_losses.png"
    if not loss_path.exists():
        figure, axis = plt.subplots(figsize=(8, 4))
        axis.plot([item["epoch"] for item in history], [item["training_loss"] for item in history], marker="o", label="Training BCE")
        axis.plot([item["epoch"] for item in history], [item["validation_loss"] for item in history], marker="o", label="Validation BCE")
        axis.set(xlabel="Epoch", ylabel="Sample-weighted BCE loss", title="Random RoBERTa: training and validation")
        axis.legend()
        axis.grid(alpha=0.2)
        figure.tight_layout()
        figure.savefig(loss_path, dpi=160)
        plt.close(figure)
    metrics_path = folder / "validation_per_label.png"
    if not metrics_path.exists():
        selected = read_json(run_dir / "validation_metrics.json")["metrics_at_validation_thresholds"]
        per_label = selected["per_label"]
        figure, axis = plt.subplots(figsize=(10, 4))
        positions = np.arange(len(LABELS))
        for offset, name in enumerate(("precision", "recall", "f1")):
            axis.bar(positions + (offset - 1) * 0.24, [per_label[label][name] for label in LABELS], width=0.24, label=name.title())
        axis.set_xticks(positions, LABELS, rotation=20)
        axis.set(ylim=(0, 1), ylabel="Score", title="Validation selection diagnostics at selected thresholds")
        axis.legend()
        axis.grid(axis="y", alpha=0.2)
        figure.tight_layout()
        figure.savefig(metrics_path, dpi=160)
        plt.close(figure)
    return [loss_path, metrics_path]


def package_results(run_dir: str | Path, *, verification_private: bool = False) -> dict[str, str]:
    """Make a shareable aggregate ZIP and an optional explicitly private ZIP."""
    destination = Path(run_dir).resolve()
    verify_frozen_run(destination)
    metadata = read_json(destination / "metadata.json")
    identity = read_json(destination / IDENTITY_FILE)
    whitelist = (
        "model_family", "initialization", "tokenizer_source", "tokenizer_revision", "synthetic", "engineering_smoke", "checkpoint_selection", "best_checkpoint", "train_positive_weights", "threshold_selection", "threshold_grid", "deterministic_algorithms", "cudnn_benchmark", "attention_implementation", "cublas_workspace_config", "amp_enabled", "effective_batch_size", "effective_batch_size_note", "parameter_count", "test_status", "protocol_note", "export_matches_best_checkpoint", "fresh_export_validation_loss_matches_best", "gradient_checkpointing", "checkpoint_every_updates", "resume_protocol", "training_completed_utc", "tested_environment_limitations",
    )
    test_evaluated = (destination / "public_test_results.json").exists()
    result_metadata = {
        "stage": "developed_and_frozen", "label_order": list(LABELS),
        "source_sha256": identity["source_sha256"],
        "code_sha256": identity["code_sha256"],
        "tokenizer_revision": identity["tokenizer_revision"],
        "test_evaluation_performed": test_evaluated,
        "validation_is_selection_diagnostic": True,
        "raw_texts_included": False, "per_row_arrays_included": False, "model_weights_included": False,
    }
    entries: dict[str, bytes | Path] = {name: destination / name for name in ("config.json", "data_summary.json", "training_history.json", "validation_metrics.json", "thresholds.json")}
    entries["run_metadata.json"] = _json_bytes({name: metadata[name] for name in whitelist if name in metadata})
    entries["versions_hardware.json"] = _json_bytes(identity["runtime"])
    entries["result_metadata.json"] = _json_bytes(result_metadata)
    entries["README.txt"] = (
        "Aggregate review results for the original randomly initialized RoBERTa.\n"
        "Validation selected the checkpoint and thresholds; its scores are selection diagnostics.\n"
        "The internal exploratory test remains unscored unless explicitly requested separately.\n"
        "The original unlabeled competition test CSV is not used.\n"
        "This ZIP has no raw comments, identifiers, per-row arrays, checkpoints, or model weights.\n"
        "Synthetic smoke results, when marked synthetic=true, are engineering checks only.\n"
    ).encode()
    for path in _review_plots(destination):
        entries["plots/" + path.name] = path
    if test_evaluated:
        entries["public_test_results.json"] = destination / "public_test_results.json"
    # Retain the original development bundle after a later explicit test. A new
    # filename prevents silently returning the old validation-only contents.
    review_name = "results_for_review_with_test.zip" if test_evaluated else "results_for_review.zip"
    output = {"results_for_review": str(_zip_once(destination / review_name, entries))}
    if verification_private:
        private_entries: dict[str, bytes | Path] = {
            "validation_predictions.npz": destination / "private" / "validation_predictions.npz",
            "splits.npz": destination / "private" / "splits.npz",
            "thresholds.json": destination / "thresholds.json",
            "config.json": destination / "config.json",
            "data_summary.json": destination / "data_summary.json",
            "validation_metrics.json": destination / "validation_metrics.json",
            "training_history.json": destination / "training_history.json",
            "PRIVATE_README.txt": b"PRIVATE VERIFICATION PACKAGE. Never publish or commit to GitHub.\nContains per-row labels/probabilities/indices, which may be sensitive.\nNo raw comments, original IDs, model weights or optimizer checkpoints.\n",
        }
        if (destination / "private" / "test_predictions.npz").exists():
            private_entries["test_predictions.npz"] = destination / "private" / "test_predictions.npz"
            private_entries["public_test_results.json"] = destination / "public_test_results.json"
        private_entries["verification_manifest.json"] = _json_bytes({
            "label_order": list(LABELS), "source_sha256": identity["source_sha256"],
            "code_sha256": identity["code_sha256"], "tokenizer_revision": identity["tokenizer_revision"],
            "runtime": identity["runtime"],
            "artifacts_sha256": {name: file_sha256(value) for name, value in private_entries.items() if isinstance(value, Path)},
        })
        private_name = "verification_private_with_test.zip" if test_evaluated else "verification_private.zip"
        output["verification_private"] = str(_zip_once(destination / private_name, private_entries))
    return output


def run_colab_training(data_path: str | Path, run_dir: str | Path, *, config: ColabConfig | None = None, resume: bool = True, evaluate: bool = False, verification_private: bool = False, quiet: bool = False, _smoke: bool = False, _stop_after_steps: int | None = None) -> dict[str, Any]:
    """Train/resume/freeze. Test scoring is a separate explicit opt-in."""
    config = config or ColabConfig()
    dependencies = core._dependencies()
    _assert_environment(dependencies, smoke=_smoke, config=config)
    device = core._set_determinism(dependencies, config.training)
    destination = Path(run_dir).resolve()
    supplied_source = Path(data_path).resolve()
    if not supplied_source.is_file():
        raise FileNotFoundError(f"CSV not found: {supplied_source}")
    existed = destination.exists()
    if not existed:
        destination = create_run_directory(destination)
        shutil.copyfile(supplied_source, destination / "private" / "source.csv")
    elif not resume:
        raise FileExistsError("Run directory exists; resume it or choose a new run directory")
    source_path = destination / "private" / "source.csv"
    identity_path = destination / IDENTITY_FILE
    tokenizer_dir = destination / TOKENIZER_DIRECTORY
    if not source_path.is_file() or file_sha256(supplied_source) != file_sha256(source_path):
        raise ValueError("Supplied CSV differs from the original private source snapshot")
    if existed and not identity_path.is_file():
        raise ValueError("Existing directory has no Colab run identity; choose a new run directory")
    identity = read_json(identity_path) if existed else None
    if identity is not None:
        _validate_identity(identity, config=config, source_path=source_path, tokenizer_dir=tokenizer_dir, dependencies=dependencies, smoke=_smoke)
        _recover_freeze_checksum(destination, identity, dependencies)
    if (destination / "manifest.json").exists() or (destination / "manifest.sha256").exists():
        verify_frozen_run(destination)
        if evaluate:
            core.evaluate_run(destination, device=config.training.device)
        history = read_json(destination / "training_history.json")
        return {
            "run_dir": str(destination), "stage": "already_frozen", "resumed": True,
            "history": history["epochs"], "best_checkpoint": history["best_checkpoint"],
            "global_optimizer_steps": history["epochs"][-1]["optimizer_steps"],
            "checkpoint_path": str(destination / LATEST_CHECKPOINT),
            "test_evaluated": (destination / "public_test_results.json").is_file(),
            **package_results(destination, verification_private=verification_private),
        }
    dataset = load_dataset(source_path, conflict_policy=config.training.conflict_policy, empty_text_policy=config.training.empty_text_policy)
    splits = split_dataset(dataset, seed=config.training.seed)
    _verify_full_data(dataset, splits, config, smoke=_smoke)
    split_fingerprint = {name: hashlib.sha256(indices.astype("<i8").tobytes()).hexdigest() for name, indices in splits.as_dict().items()}
    positive_weights = core.validate_training_targets(dataset.targets[splits.train])
    if identity is not None:
        if identity["split_sha256"] != split_fingerprint:
            raise ValueError("Reconstructed split differs from the saved run identity")
        tokenizer = dependencies.AutoTokenizer.from_pretrained(tokenizer_dir, local_files_only=True, use_fast=True)
        revision = identity["tokenizer_revision"]
    else:
        tokenizer, revision = _new_tokenizer(dataset, splits, config, dependencies, tokenizer_dir, smoke=_smoke)
        identity = {
            "schema_version": CHECKPOINT_SCHEMA, "requested_config": asdict(config),
            "source_sha256": dataset.source_sha256, "split_sha256": split_fingerprint,
            "code_sha256": _source_code_hashes(), "tokenizer_sha256": _directory_hashes(tokenizer_dir),
            "tokenizer_revision": revision, "runtime": _runtime_identity(dependencies), "smoke": _smoke,
        }
        _atomic_json(identity_path, identity)
    model = core._new_model(tokenizer, dependencies, smoke=_smoke).to(device)
    if config.training.max_length + tokenizer.pad_token_id + 1 > model.config.max_position_embeddings:
        raise ValueError("max_length exceeds the configured RoBERTa position capacity")
    if config.gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    criterion = dependencies.torch.nn.BCEWithLogitsLoss(pos_weight=dependencies.torch.tensor(positive_weights, dtype=dependencies.torch.float32, device=device))
    state = _train_resumable(model, dataset, splits, tokenizer, criterion, config, dependencies, device, destination, resume=resume, quiet=quiet, stop_after_steps=_stop_after_steps)
    _validate_identity(identity, config=config, source_path=source_path, tokenizer_dir=tokenizer_dir, dependencies=dependencies, smoke=_smoke)
    if state["interrupted"]:
        return {"run_dir": str(destination), "stage": "interrupted", "checkpoint_path": str(destination / LATEST_CHECKPOINT), "global_optimizer_steps": state["global_optimizer_steps"], "history": state["history"], "best_checkpoint": state["best"], "resumed": state["resumed"]}
    # Recoverable export: only manifest-free derived files are removed. Private
    # source, tokenizer, latest optimizer state and best weights remain intact.
    if (destination / "model").exists():
        shutil.rmtree(destination / "model")
    for name in ("config.json", "data_summary.json", "metadata.json", "thresholds.json", "training_history.json", "validation_metrics.json", "private/splits.npz", "private/validation_predictions.npz"):
        (destination / name).unlink(missing_ok=True)
    model, tokenizer, best = core._export_best(model, tokenizer, destination / BEST_CHECKPOINT, destination / "model", dependencies, device)
    validation_loader = core._loader(dataset, splits.validation, tokenizer, config.training, dependencies, shuffle=False)
    fresh_loss, targets, probabilities = core._predict(model, validation_loader, dependencies, device, criterion=criterion, amp_enabled=config.training.amp and device.type == "cuda")
    if not np.array_equal(targets, dataset.targets[splits.validation]):
        raise RuntimeError("Fresh exported validation inference changed row/label order")
    if not np.isclose(fresh_loss, best["validation_loss"], rtol=1e-5, atol=1e-6):
        raise RuntimeError("Fresh best export does not reproduce the selected validation loss")
    thresholds = select_thresholds(targets, probabilities)
    validation_path = destination / "private" / "validation_predictions.npz"
    np.savez_compressed(validation_path, targets=targets, probabilities=probabilities, row_indices=splits.validation)
    history = {"epochs": state["history"], "best_checkpoint": best, "fresh_validation_loss": fresh_loss}
    _atomic_json(destination / "training_history.json", history)
    _atomic_json(destination / "validation_metrics.json", {
        "split": "validation", "synthetic": bool(dataset.summary["synthetic"]),
        "metrics_at_0_5": evaluate_predictions(targets, probabilities, np.full(len(LABELS), 0.5)),
        "metrics_at_validation_thresholds": evaluate_predictions(targets, probabilities, thresholds),
        "selection_note": "Checkpoint and thresholds were selected using validation. Scores are selection diagnostics; the internal test was not used.",
    })
    metadata = {
        **runtime_metadata(config.training.seed),
        "model_family": "custom RoBERTa sequence classifier",
        "initialization": "random weights; no pretrained model weights",
        "tokenizer_source": "locally constructed train-only synthetic tokenizer" if _smoke else "roberta-base tokenizer only",
        "tokenizer_revision": revision,
        "synthetic": bool(dataset.summary["synthetic"]), "engineering_smoke": _smoke,
        "checkpoint_selection": "minimum sample-weighted validation BCE loss", "best_checkpoint": best,
        "train_positive_weights": dict(zip(LABELS, positive_weights.tolist())),
        "threshold_selection": "per-label validation F1; >= comparator; ties nearest 0.5 then larger; missing class keeps 0.5",
        "threshold_grid": list(DEFAULT_THRESHOLD_GRID), "deterministic_algorithms": True,
        "cudnn_benchmark": False, "attention_implementation": "eager", "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "amp_enabled": config.training.amp and device.type == "cuda",
        "effective_batch_size": config.training.batch_size * config.training.accumulation_steps,
        "effective_batch_size_note": "Nominal maximum; each accumulation group is weighted by its actual sample count.",
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "test_status": "unscored during development",
        "protocol_note": "New internal exploratory holdout; the historical cohort was previously examined, so this is not a pristine external test.",
        "export_matches_best_checkpoint": True, "fresh_export_validation_loss_matches_best": True,
        "gradient_checkpointing": config.gradient_checkpointing, "checkpoint_every_updates": config.checkpoint_every_updates,
        "resume_protocol": "Exact code/config/source/tokenizer/software/hardware fingerprints; accumulation boundaries; global RNG and deterministic per-epoch loader restored.",
        "runner_code_sha256": identity["code_sha256"]["scripts/colab_train.py"],
        "training_completed_utc": runtime_metadata(config.training.seed)["created_utc"],
        "tested_environment_limitations": "Python 3.12 and torch 2.8.0+cpu: tiny offline mechanics only. Preserve Colab CUDA torch >=2.8; CUDA execution, memory and full-data model quality must be verified in the user's runtime. Exact continuation requires the recorded versions and GPU family/capability.",
    }
    architecture = {name: getattr(model.config, name) for name in core.ORIGINAL_ARCHITECTURE}
    _validate_identity(identity, config=config, source_path=source_path, tokenizer_dir=tokenizer_dir, dependencies=dependencies, smoke=_smoke)
    if not state["training_complete"] or (destination / TEST_MARKER).exists():
        raise ValueError("Freezing requires completed training before any test access")
    _atomic_json(destination / FREEZE_TRANSACTION, {
        "schema_version": 1,
        "identity_sha256": file_sha256(identity_path),
        "latest_checkpoint_sha256": file_sha256(destination / LATEST_CHECKPOINT),
        "best_checkpoint_sha256": file_sha256(destination / BEST_CHECKPOINT),
    })
    freeze_run(
        destination,
        config={"method": "random_initialized_roberta", "training": asdict(config.training), "architecture": architecture, "label_order": list(LABELS), "colab": {"gradient_checkpointing": config.gradient_checkpointing, "checkpoint_every_updates": config.checkpoint_every_updates, "require_full_dataset": config.require_full_dataset}},
        data_summary={**dataset.summary, "splits": split_summary(dataset, splits)},
        source_path=source_path, model_path=destination / "model", thresholds=thresholds, splits=splits, metadata=metadata,
        extra_artifacts=[destination / relative for relative in FREEZE_EXTRA_ARTIFACTS],
    )
    verify_frozen_run(destination)
    (destination / FREEZE_TRANSACTION).unlink()
    if evaluate:
        core.evaluate_run(destination, device=config.training.device)
    return {
        "run_dir": str(destination), "stage": "developed_and_frozen", "resumed": state["resumed"],
        "global_optimizer_steps": state["global_optimizer_steps"], "history": state["history"], "best_checkpoint": best,
        "checkpoint_path": str(destination / LATEST_CHECKPOINT), "test_evaluated": evaluate,
        **package_results(destination, verification_private=verification_private),
    }


def run_smoke(run_dir: str | Path) -> dict[str, Any]:
    """Offline interrupt/resume check, including an epoch and partial groups."""
    parent = Path(run_dir).resolve()
    parent.mkdir(parents=True, exist_ok=False)
    source = create_synthetic_fixture(parent / "synthetic.csv", rows=240, seed=42)
    config = ColabConfig(training=core.TrainConfig(
        epochs=2, patience=2, batch_size=13, accumulation_steps=5,
        learning_rate=1e-3, max_length=16, cpu_threads=1,
        device="cpu", amp=False, local_files_only=True,
    ), checkpoint_every_updates=1, require_full_dataset=False)
    continuous = run_colab_training(source, parent / "continuous", config=config, quiet=True, _smoke=True)
    interrupted = run_colab_training(source, parent / "resumed", config=config, quiet=True, _smoke=True, _stop_after_steps=2)
    resumed = run_colab_training(source, parent / "resumed", config=config, quiet=True, _smoke=True)
    dependencies = core._dependencies()
    first = dependencies.torch.load(parent / "continuous" / LATEST_CHECKPOINT, map_location="cpu", weights_only=True)
    second = dependencies.torch.load(parent / "resumed" / LATEST_CHECKPOINT, map_location="cpu", weights_only=True)
    exact_weights = all(dependencies.torch.equal(value, second["model_state_dict"][name]) for name, value in first["model_state_dict"].items())
    with np.load(parent / "continuous" / "private" / "validation_predictions.npz", allow_pickle=False) as one, np.load(parent / "resumed" / "private" / "validation_predictions.npz", allow_pickle=False) as two:
        exact_probabilities = np.array_equal(one["probabilities"], two["probabilities"])
    if not exact_weights or not exact_probabilities or continuous["history"] != resumed["history"]:
        raise RuntimeError("Interrupted/resumed smoke differs from uninterrupted training")
    return {
        "status": "passed", "synthetic": True, "purpose": "CPU engineering mechanics only; no real-data or CUDA performance claim",
        "run_dir": str(parent), "interrupted_at_updates": interrupted["global_optimizer_steps"],
        "total_optimizer_updates": resumed["global_optimizer_steps"],
        "checks": {"exact_final_weights": exact_weights, "exact_exported_validation_probabilities": exact_probabilities, "exact_history": True, "epoch_and_partial_accumulation_groups": True, "test_unscored": not resumed["test_evaluated"], "local_train_only_tokenizer": True, "safe_weights_only_checkpoint": True},
    }


def _parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    stages = parser.add_subparsers(dest="stage", required=True)
    develop = stages.add_parser("develop", help="Train/resume the full original GPU experiment; test stays unscored")
    develop.add_argument("--data", required=True, type=Path)
    develop.add_argument("--run-dir", required=True, type=Path)
    develop.add_argument("--gradient-checkpointing", action="store_true", help="Explicit memory-saving choice recorded in config; cannot change during resume")
    develop.add_argument("--evaluate", action="store_true", help="Opt in to the frozen single-use internal test AFTER development")
    develop.add_argument("--verification-private", action="store_true", help="Also make a private per-row verification ZIP; never publish it")
    evaluate = stages.add_parser("evaluate", help="Explicitly score a frozen run once, using core test guards")
    evaluate.add_argument("--run-dir", required=True, type=Path)
    evaluate.add_argument("--device", choices=("cpu", "cuda"), default="cuda")
    smoke = stages.add_parser("smoke", help="Tiny offline CPU exact-resume mechanics, never real-data training")
    smoke.add_argument("--run-dir", required=True, type=Path)
    return parser


def main(argv: list[str] | None = None) -> int:
    parser = _parser()
    args = parser.parse_args(argv)
    try:
        if args.stage == "smoke":
            result = run_smoke(args.run_dir)
        elif args.stage == "evaluate":
            result = {"evaluation": core.evaluate_run(args.run_dir, device=args.device), **package_results(args.run_dir)}
        else:
            result = run_colab_training(args.data, args.run_dir, config=ColabConfig(gradient_checkpointing=args.gradient_checkpointing), evaluate=args.evaluate, verification_private=args.verification_private)
    except (ImportError, OSError, ValueError, RuntimeError, FloatingPointError) as error:
        parser.exit(2, f"colab: {error}\n")
    print(json.dumps(result, indent=2, allow_nan=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/toxic-comment-classification/requirements-colab.txt
# Python 3.12+. Preserve Colab's CUDA-enabled torch >=2.8; do not replace it.
# CPU mechanics tested with torch 2.8.0+cpu. CUDA training is not validated here.
# Standalone copy of the core pins so the notebook needs no other requirements file.
numpy==2.3.5
pandas==2.2.3
scipy==1.17.0
scikit-learn==1.8.0
joblib==1.5.3
transformers==4.57.1
tokenizers==0.22.2
safetensors==0.8.0
huggingface-hub==0.36.2
matplotlib==3.10.8


## 3. Verify source and install the other pinned dependencies

Torch is deliberately absent from this requirements file. Installation changes
only the listed non-torch dependencies and their dependencies. If Colab reports
that already imported packages need a session restart, restart the session and
rerun from cell 1. Persisted Drive files and checkpoints remain available.

In [ ]:
import hashlib
EMBEDDED_FILES = ('src/toxic_comments/__init__.py', 'src/toxic_comments/artifacts.py', 'src/toxic_comments/baseline.py', 'src/toxic_comments/data.py', 'src/toxic_comments/metrics.py', 'src/toxic_comments/neural.py', 'scripts/colab_train.py', 'requirements-colab.txt')
EMBEDDED_SHA256 = {'src/toxic_comments/__init__.py': 'ed7fe61628007bb1ea08a66c9230e16697bc1cac0b2fe160ee5c4a372f13e5aa', 'src/toxic_comments/artifacts.py': '7a95604d6bb4113e664e8a6c443203018396c7a2a9a718be7746e0bc9d278221', 'src/toxic_comments/baseline.py': 'b95ebdaf7a42aaeb97d13d0a2f1ccf0069407c193a6ad4a5929c4b1634b1facf', 'src/toxic_comments/data.py': '3258b13dbe200333174c39cb36085268ce7d70b8a8d7185cf01f150d664f460f', 'src/toxic_comments/metrics.py': 'de4f9d130179c03354414ee5825bf6b52a6a88926f45596cc2070b9224b75b38', 'src/toxic_comments/neural.py': 'd96fe7a1624d05bb32aac84e3279d8f1081b7e834248ba7fa7488c7fb850cac3', 'scripts/colab_train.py': '09c0afb5f4b2bca9431922cfda1b54eaadcb7f42d105fdd3be74e8d794eb746d', 'requirements-colab.txt': '211eceaed3373276e9a06c281d7288d1926ec567d528e82ae57f08e90cf1fcd2'}
for relative in EMBEDDED_FILES:
    observed = hashlib.sha256((PROJECT_DIR / relative).read_bytes()).hexdigest()
    if observed != EMBEDDED_SHA256[relative]:
        raise RuntimeError(f'Embedded source changed: {relative}')
print('All eight embedded source files match their manifest.')


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)


In [ ]:
import importlib.metadata

expected_versions = {}
for line in (PROJECT_DIR / "requirements-colab.txt").read_text().splitlines():
    if "==" in line and not line.lstrip().startswith("#"):
        name, version = line.strip().split("==", 1)
        expected_versions[name] = version
for name, version in expected_versions.items():
    observed = importlib.metadata.version(name)
    if observed != version:
        raise RuntimeError(f"Installed {name} {observed}; expected {version}.")
module_names = {"scikit-learn": "sklearn", "huggingface-hub": "huggingface_hub"}
for name, version in expected_versions.items():
    imported_module = sys.modules.get(module_names.get(name, name))
    loaded_version = getattr(imported_module, "__version__", None)
    if loaded_version is not None and str(loaded_version) != version:
        raise RuntimeError(f"{name} {loaded_version} is still imported after installing {version}. "
                           "Restart the Colab session, then rerun this notebook from cell 1.")
if importlib.metadata.version("torch") != COLAB_TORCH_DISTRIBUTION_VERSION:
    raise RuntimeError("The torch build changed. Reconnect to a compatible CUDA runtime and rerun setup.")

sys.path.insert(0, str(PROJECT_DIR / "src"))
sys.path.insert(0, str(PROJECT_DIR))
from scripts import colab_train as colab_runner
from toxic_comments import neural
dependencies = neural._dependencies()
colab_runner._assert_environment(dependencies, smoke=False, config=colab_runner.ColabConfig())
print("Pinned environment imported successfully; Colab's CUDA torch was preserved.")
print(expected_versions)


## 4. Mount Drive and select the original training CSV

The **run folder must stay on Google Drive** so a runtime disconnect can resume
from its last complete checkpoint. Keep the same `RUN_NAME` to resume; choose a
new name for a separate experiment. Use a plain folder name without slashes.

Leave `CSV_ON_DRIVE` empty to upload the CSV once. Subsequent sessions reuse the
saved run's private source snapshot or the remembered input. To use a file already
on Drive, enter its full `/content/drive/MyDrive/.../train_preprocessed.csv` path.
The CSV is copied privately into the run and never included in the review ZIP.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/toxic-comment-classification" # @param {type:"string"}
RUN_NAME = "random-roberta-seed42" # @param {type:"string"}
CSV_ON_DRIVE = "" # @param {type:"string"}
if not RUN_NAME or RUN_NAME in {".", ".."} or "/" in RUN_NAME or "\\" in RUN_NAME:
    raise ValueError("RUN_NAME must be one plain folder name.")
PERSISTENT_ROOT = Path(DRIVE_PROJECT_DIR).expanduser().resolve()
if not PERSISTENT_ROOT.is_relative_to(Path("/content/drive/MyDrive")):
    raise ValueError("Use a persistent folder under /content/drive/MyDrive.")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = PERSISTENT_ROOT / "runs" / RUN_NAME
run_source = RUN_DIR / "private" / "source.csv"
remembered_source = PERSISTENT_ROOT / "inputs" / "train_preprocessed.csv"
if CSV_ON_DRIVE.strip():
    DATA_FILE = Path(CSV_ON_DRIVE).expanduser().resolve()
elif run_source.is_file():
    DATA_FILE = run_source
elif remembered_source.is_file():
    DATA_FILE = remembered_source
else:
    uploaded = files.upload()
    if len(uploaded) != 1 or "train_preprocessed.csv" not in uploaded:
        raise ValueError("Upload only the original train_preprocessed.csv, with its original filename.")
    csv_bytes = uploaded["train_preprocessed.csv"]
    if hashlib.sha256(csv_bytes).hexdigest() != colab_runner.FULL_SOURCE_SHA256:
        raise ValueError("Uploaded bytes differ from the original supplied training CSV.")
    remembered_source.parent.mkdir(parents=True, exist_ok=True)
    remembered_source.write_bytes(csv_bytes)
    del uploaded, csv_bytes
    DATA_FILE = remembered_source
if not DATA_FILE.is_file():
    raise FileNotFoundError("Training CSV not found at the chosen Drive path.")
if hashlib.sha256(DATA_FILE.read_bytes()).hexdigest() != colab_runner.FULL_SOURCE_SHA256:
    raise ValueError("The selected CSV is not the original supplied train_preprocessed.csv.")
print({"source_verified": True, "run_name": RUN_NAME, "existing_run": RUN_DIR.exists(),
       "persistent_checkpoints": True})


## 5. Audit the complete cohort and show the training plan

This reads the complete CSV and prints aggregate counts only. Row counts, source
hash, empty-text policy, conflicting annotations, and split counts must all match
the supplied dataset before training begins. No raw comments or identifiers are
displayed. The runner performs the same checks again at run creation/resume.

Gradient checkpointing is an optional memory-saving implementation choice. The
default is False. To enable it, do so **before a new run**; it cannot change during
resume. The original architecture, sequence length, and cohort remain fixed.

In [ ]:
import math
from dataclasses import asdict
from toxic_comments.data import load_dataset, split_dataset

GRADIENT_CHECKPOINTING = False # @param {type:"boolean"}
CONFIG = colab_runner.ColabConfig(gradient_checkpointing=GRADIENT_CHECKPOINTING)
audit_dataset = load_dataset(DATA_FILE, conflict_policy="keep", empty_text_policy="drop")
audit_splits = split_dataset(audit_dataset, seed=42)
colab_runner._verify_full_data(audit_dataset, audit_splits, CONFIG, smoke=False)
print({"input_rows": audit_dataset.summary["input_rows"],
       "eligible_rows": audit_dataset.summary["retained_rows"],
       "excluded_empty_text_rows": audit_dataset.summary["excluded_empty_text_rows"],
       "conflicting_label_groups": audit_dataset.summary["conflicting_label_groups"],
       "split_rows": {name: len(indices) for name, indices in audit_splits.as_dict().items()},
       "duplicate_group_overlap": 0})
updates_per_epoch = math.ceil(math.ceil(len(audit_splits.train) / CONFIG.training.batch_size)
                              / CONFIG.training.accumulation_steps)
print({"label_order": list(colab_runner.LABELS),
       "random_architecture": neural.ORIGINAL_ARCHITECTURE,
       "expected_parameters": 51484678, "training": asdict(CONFIG.training),
       "planned_updates_per_epoch": updates_per_epoch,
       "maximum_optimizer_updates": updates_per_epoch * CONFIG.training.epochs,
       "checkpoint_every_updates": CONFIG.checkpoint_every_updates,
       "gradient_checkpointing": CONFIG.gradient_checkpointing})
del audit_dataset, audit_splits


## 6. Train or resume the original model

This is the expensive cell: it trains the full eligible cohort on the GPU.
There is no hidden row cap or CPU fallback. The loop prints progress at least
about every 30 seconds and saves at every 100 actual optimizer updates plus each
epoch boundary. Recovered work starts at the saved accumulation boundary; at
most the unsaved interval must be replayed after a disconnect.

The atomic `private/latest_checkpoint.pt` contains the model, AdamW optimizer,
cosine scheduler, AMP scaler, Python/NumPy/torch CPU/CUDA RNG, deterministic epoch
shuffle offset, history, and patience state. `private/best_checkpoint.pt` retains
the lowest validation-loss model. Exact continuation requires the **same source
bytes, code, configuration, tokenizer snapshot, software versions, torch CUDA
build, GPU family/capability/memory, and GPU count**. A mismatch fails explicitly;
never edit the saved identity to bypass it.

If disconnected, reopen this same notebook, reconnect to a compatible GPU,
rerun setup, mount Drive, retain `RUN_NAME` and configuration, then rerun this
cell. Full training runs in a fresh Python subprocess so dependencies replaced
by pip cannot leave stale notebook imports in the training process. No tokenizer
download is needed when resuming its saved local snapshot.
Torch checkpoints should come only from your own trusted run folder.

In [ ]:
import json

# Fresh process; no --evaluate flag. Internal-test scoring stays disabled.
training_command = [sys.executable, "-u", str(PROJECT_DIR / "scripts/colab_train.py"),
                    "develop", "--data", str(DATA_FILE), "--run-dir", str(RUN_DIR)]
if CONFIG.gradient_checkpointing:
    training_command.append("--gradient-checkpointing")
subprocess.run(training_command, cwd=PROJECT_DIR, check=True)
saved_history = json.loads((RUN_DIR / "training_history.json").read_text())
print({"training_completed": True,
       "global_optimizer_steps": saved_history["epochs"][-1]["optimizer_steps"],
       "best_checkpoint": saved_history["best_checkpoint"],
       "test_evaluated": (RUN_DIR / "public_test_results.json").is_file()})


## 7. Inspect aggregate results and prepare the return ZIP

The exported model is reloaded from the best checkpoint before fresh validation
predictions and threshold selection. These are newly run validation selection
diagnostics; compare them with the existing CPU baseline only under this repaired
protocol. They are separate from historical rounded neural notebook logs.

`results_for_review.zip` contains aggregate metrics, thresholds, loss history,
plots, configuration, and versions/hardware. It excludes raw comments, IDs,
per-row arrays, and model/optimizer weights. Set the optional private switch
True only if row-level numerical verification is needed; its separate ZIP has
labels/probabilities/split indices, **no raw texts, original IDs, or weights**.
Never commit or publicly share that private ZIP.

In [ ]:
import json
from IPython.display import display, Image

MAKE_PRIVATE_VERIFICATION = False # @param {type:"boolean"}
review_packages = colab_runner.package_results(
    RUN_DIR, verification_private=MAKE_PRIVATE_VERIFICATION,
)
validation = json.loads((RUN_DIR / "validation_metrics.json").read_text())
selected_metrics = validation["metrics_at_validation_thresholds"]
print(json.dumps({"split": "validation selection diagnostics",
                  "metrics": selected_metrics,
                  "test_evaluated": (RUN_DIR / "public_test_results.json").is_file()}, indent=2))
for plot in sorted((RUN_DIR / "review_plots").glob("*.png")):
    display(Image(filename=str(plot)))
print({name: Path(path).name for name, path in review_packages.items()})


## 8. Optional internal exploratory test — keep False initially

Return the development ZIP first so checkpoint/threshold results can be reviewed
before improvement decisions. Only turn this switch True for a separately
requested frozen evaluation. It scores the 23,955-row internal split once using
the frozen best export and validation thresholds; it never scores the unlabeled
competition CSV. The core records a single-use evaluation marker before access.
An interrupted test must not be silently rerun.

A later explicit test creates `results_for_review_with_test.zip` and, if requested,
`verification_private_with_test.zip`. The original development bundles are kept.
Changing the switch back to False cannot undo an evaluation already performed.

In [ ]:
START_INTERNAL_TEST = False # @param {type:"boolean"}
if START_INTERNAL_TEST:
    if (RUN_DIR / "public_test_results.json").is_file():
        test_result = json.loads((RUN_DIR / "public_test_results.json").read_text())
        print("Internal test already completed; displaying the saved aggregate result.")
    else:
        test_result = neural.evaluate_run(RUN_DIR, device="cuda")
    print(json.dumps(test_result, indent=2))
    review_packages = colab_runner.package_results(
        RUN_DIR, verification_private=MAKE_PRIVATE_VERIFICATION,
    )
else:
    print("No new internal-test evaluation requested. Return the aggregate development ZIP first.")


## 9. Download the aggregate result and return it for review

Upload **`results_for_review.zip`** back to this conversation after training.
If a test was explicitly requested, return the bundle with `_with_test` in its
name. The optional private numerical bundle is for private review only. Leave
CSV data, checkpoints, and model weights in your Drive run folder.

In [ ]:
files.download(review_packages["results_for_review"])
if MAKE_PRIVATE_VERIFICATION:
    files.download(review_packages["verification_private"])


## Troubleshooting and verification limits

- **No GPU / old Python or torch:** choose a compatible Colab runtime and rerun
  setup. Do not replace its CUDA torch with the CPU wheel used by local tests.
- **Dependency import failure after installation:** restart the session and
  rerun setup. The notebook verifies all pinned versions before training.
- **Out of memory:** end the failed attempt. Enable gradient checkpointing for
  a **new run name** and keep all original experiment settings. Do not silently
  reduce the cohort, length, architecture, or accumulation settings.
- **Resume identity mismatch:** reconnect to the recorded software/GPU family
  or start a new clearly separate run. Do not rewrite identity/checkpoint files.
- **Drive serialization error or disconnect:** reconnect and resume. Only the
  last complete checkpoint is usable; durability also depends on Google Drive.
- **Source mismatch:** use the original CSV bytes. Resaving a CSV changes its
  SHA-256 even when displayed values appear identical.

Locally verified: exact embedded source and notebook structure, tiny **offline
CPU synthetic** training, mid-epoch interruption/resume, best export and fresh
validation, numerical checkpoint/array consistency, and test/privacy guards.
The full real-data model, Colab services, CUDA execution, memory use, and final
model quality have **not** been exercised locally. Do not treat CPU synthetic
scores as model-quality evidence or guarantee GPU wall time.